In [392]:
import pandas as pd
import os
import gzip
import json
from datetime import datetime
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

In [393]:

folder_path = r'C:\Users\programming.com\Downloads\dtc_UserAttributes'

# Get the list of all files in the folder
file_list = os.listdir(folder_path)

# Print the names of the files
files = [file_name for file_name in file_list]
files

['6boerkskyqy3jb4na6g6i7n6yq.json.gz',
 '6jbrthn5sm4w3ktgtmop4lknpi.json.gz',
 'b3xyfq7zxqyfhkrjx5ggnrusg4.json.gz',
 'fvxrw7amrm7zhn5v5qwnr5disu.json.gz']

In [394]:
# Initialize an empty list to store DataFrames
dataframes = []
file_paths = []
# Loop through each file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.json.gz'):
        file_path = os.path.join(folder_path, file_name)
        file_paths.append(file_path)
        

# Function to flatten the nested structure in the 'Item' column
def flatten_item_column(df):
    # Ensure the 'Item' column is converted to a DataFrame
    if 'Item' in df.columns:
        expanded = pd.json_normalize(df['Item'])  # Flatten the nested JSON in 'Item'
        return pd.concat([df.drop(columns=['Item']), expanded], axis=1)
    return df

# Loop through each file path and process the .json.gz files
for file_path in file_paths:
    try:
        # Read the JSON file into a DataFrame
        df = pd.read_json(file_path, compression='gzip', lines=True)
        
        # Flatten the 'Item' column if it exists
        df = flatten_item_column(df)
        
        dataframes.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

In [395]:

# Combine all DataFrames into a single DataFrame
user_attributes_df = pd.concat(dataframes, ignore_index=True)

# Display the combined DataFrame
user_attributes_df.head()

,version.N,user_id.S,sexual_orientation.S,time_away_from_home.S,date_of_birth.S,year_of_birth.N,living_situation.L,group2.S,gender_identity.S,nickname.S,...,last_modified_ts.S,user_type.S,deactivation_ts.S,last_modified_by.S,deactivation_user_name.S,date_of_birth.NULL,year_of_birth.NULL,clinician_provider_id.S,notes.S,dt_complete.S
0,0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,straight_or_heterosexual,yes_part_time,1991-07-27,1991,[{'S': 'spouse_or_partner'}],000,male,RTM-000-0010,...,2024-08-22T15:14:34.703734+00:00,deactivated,2024-08-22T15:14:34.686512+00:00,disable-endpoint,,NaN,NaN,NaN,NaN,NaN
1,1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN,NaN,000,NaN,RTM-000-0010,...,2024-07-19T15:27:13.009489+00:00,prospect,NaN,User Ingestion API,NaN,True,True,NaN,NaN,NaN
2,2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN,NaN,000,NaN,RTM-000-0010,...,2024-07-19T15:27:15.203724+00:00,invited,NaN,POST v2/usr/invite,NaN,True,True,NaN,NaN,NaN
3,3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN,NaN,000,NaN,RTM-000-0010,...,2024-07-19T15:38:24.438300+00:00,invited,NaN,taylor@healthrhythms.com,NaN,True,True,NaN,NaN,NaN
4,4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,straight_or_heterosexual,yes_part_time,1991-07-27,1991,[{'S': 'spouse_or_partner'}],000,male,RTM-000-0010,...,2024-07-20T15:44:13.687669+00:00,invited,NaN,PUT v2/usr/attributes by us-east-1:db841a21-c4...,NaN,NaN,NaN,NaN,NaN,NaN


In [396]:
user_attributes_df.shape

(404, 31)

In [397]:
# Clean column names by removing everything after the first dot
def clean_column_names(df):
    df.columns = [col.split('.')[0] for col in df.columns]  # Remove everything after the first dot
    return df

# Apply this cleaning function after combining the DataFrames
user_attributes_df = pd.concat(dataframes, ignore_index=True).sort_index(axis=1, ascending=True)
user_attributes_df = clean_column_names(user_attributes_df)

user_attributes_df.head(3)

,clinician_provider_id,communication_preference,date_of_birth,date_of_birth,deactivation_status,deactivation_ts,deactivation_user,deactivation_user_name,dt_complete,ethnicity,...,phone_number,phone_usage,sex_at_birth,sexual_orientation,time_away_from_home,user_id,user_type,version,year_of_birth,year_of_birth
0,NaN,[{'S': 'email'}],NaN,1991-07-27,True,2024-08-22T15:14:34.686512+00:00,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,,NaN,[{'S': 'asian'}],...,+1 (555) 555-5555,always,male,straight_or_heterosexual,yes_part_time,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,deactivated,0,1991,NaN
1,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,+1 (555) 555-5555,NaN,NaN,NaN,NaN,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,prospect,1,NaN,True
2,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,+1 (555) 555-5555,NaN,NaN,NaN,NaN,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,invited,2,NaN,True


In [398]:
user_attributes_df.shape

(404, 31)

In [399]:
user_attributes_df.columns

Index(['clinician_provider_id', 'communication_preference', 'date_of_birth',
       'date_of_birth', 'deactivation_status', 'deactivation_ts',
       'deactivation_user', 'deactivation_user_name', 'dt_complete',
       'ethnicity', 'event_time_utc_ts', 'first_name', 'gender_identity',
       'group1', 'group2', 'last_modified_by', 'last_modified_ts', 'last_name',
       'living_situation', 'nickname', 'notes', 'phone_number', 'phone_usage',
       'sex_at_birth', 'sexual_orientation', 'time_away_from_home', 'user_id',
       'user_type', 'version', 'year_of_birth', 'year_of_birth'],
      dtype='object')

In [400]:
user_attributes_df.isna().sum().sort_values(ascending=False)

dt_complete                 403
deactivation_status         392
deactivation_ts             392
deactivation_user           392
deactivation_user_name      392
notes                       389
year_of_birth               312
date_of_birth               312
living_situation            267
time_away_from_home         267
ethnicity                   267
sexual_orientation          267
sex_at_birth                267
gender_identity             267
phone_usage                 267
communication_preference    267
clinician_provider_id       154
event_time_utc_ts           116
date_of_birth                92
year_of_birth                92
last_name                     0
last_modified_ts              0
nickname                      0
group2                        0
phone_number                  0
group1                        0
first_name                    0
user_id                       0
user_type                     0
version                       0
last_modified_by              0
dtype: i

In [401]:
user_attributes_df.isna().sum().sort_index(ascending=True)

clinician_provider_id       154
communication_preference    267
date_of_birth               312
date_of_birth                92
deactivation_status         392
deactivation_ts             392
deactivation_user           392
deactivation_user_name      392
dt_complete                 403
ethnicity                   267
event_time_utc_ts           116
first_name                    0
gender_identity             267
group1                        0
group2                        0
last_modified_by              0
last_modified_ts              0
last_name                     0
living_situation            267
nickname                      0
notes                       389
phone_number                  0
phone_usage                 267
sex_at_birth                267
sexual_orientation          267
time_away_from_home         267
user_id                       0
user_type                     0
version                       0
year_of_birth                92
year_of_birth               312
dtype: i

In [402]:
user_attributes_df['date_of_birth'].isna().sum()

date_of_birth    312
date_of_birth     92
dtype: int64

In [403]:
# Get the list of columns
columns = user_attributes_df.columns
# Rename the columns with similar names 'json'
new_columns = []
date_of_birth_count = 1
for col in columns:
    if 'date_of_birth' in col:
        new_columns.append(f'date_of_birth_{date_of_birth_count}')
        date_of_birth_count += 1
    else:
        new_columns.append(col)
        
user_attributes_df.columns = new_columns

In [404]:
# Get the list of columns
columns = user_attributes_df.columns
# Rename the columns with similar names 'json'
new_columns = []
year_of_birth_count = 1
for col in columns:
    if 'year_of_birth' in col:
        new_columns.append(f'year_of_birth_{year_of_birth_count}')
        year_of_birth_count += 1
    else:
        new_columns.append(col)
        
user_attributes_df.columns = new_columns
user_attributes_df.isna().sum().sort_index(ascending=False)

year_of_birth_2             312
year_of_birth_1              92
version                       0
user_type                     0
user_id                       0
time_away_from_home         267
sexual_orientation          267
sex_at_birth                267
phone_usage                 267
phone_number                  0
notes                       389
nickname                      0
living_situation            267
last_name                     0
last_modified_ts              0
last_modified_by              0
group2                        0
group1                        0
gender_identity             267
first_name                    0
event_time_utc_ts           116
ethnicity                   267
dt_complete                 403
deactivation_user_name      392
deactivation_user           392
deactivation_ts             392
deactivation_status         392
date_of_birth_2              92
date_of_birth_1             312
communication_preference    267
clinician_provider_id       154
dtype: i

In [405]:
user_attributes_df.columns

Index(['clinician_provider_id', 'communication_preference', 'date_of_birth_1',
       'date_of_birth_2', 'deactivation_status', 'deactivation_ts',
       'deactivation_user', 'deactivation_user_name', 'dt_complete',
       'ethnicity', 'event_time_utc_ts', 'first_name', 'gender_identity',
       'group1', 'group2', 'last_modified_by', 'last_modified_ts', 'last_name',
       'living_situation', 'nickname', 'notes', 'phone_number', 'phone_usage',
       'sex_at_birth', 'sexual_orientation', 'time_away_from_home', 'user_id',
       'user_type', 'version', 'year_of_birth_1', 'year_of_birth_2'],
      dtype='object')

In [406]:
user_attributes_df[['user_id', 'ethnicity', 'gender_identity', 'sex_at_birth', 'sexual_orientation']]

,user_id,ethnicity,gender_identity,sex_at_birth,sexual_orientation
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,[{'S': 'asian'}],male,male,straight_or_heterosexual
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,[{'S': 'asian'}],male,male,straight_or_heterosexual
...,...,...,...,...,...
399,us-east-1:db841a21-c42b-ca48-2634-02a92db6d920,NaN,NaN,NaN,NaN
400,us-east-1:db841a21-c42b-ca48-2634-02a92db6d920,NaN,NaN,NaN,NaN
401,us-east-1:db841a21-c4f3-c260-a350-3e9e8ad7c935,NaN,NaN,NaN,NaN
402,us-east-1:db841a21-c4f3-c260-a350-3e9e8ad7c935,NaN,NaN,NaN,NaN


In [407]:
# Calculate the missing value ratio for each column
missing_val = (user_attributes_df.isna().sum() / user_attributes_df.shape[0]) * 100
missing_val.sort_values(ascending = False)

dt_complete                 99.752475
deactivation_status         97.029703
deactivation_ts             97.029703
deactivation_user           97.029703
deactivation_user_name      97.029703
notes                       96.287129
year_of_birth_2             77.227723
date_of_birth_1             77.227723
living_situation            66.089109
time_away_from_home         66.089109
ethnicity                   66.089109
sexual_orientation          66.089109
sex_at_birth                66.089109
gender_identity             66.089109
phone_usage                 66.089109
communication_preference    66.089109
clinician_provider_id       38.118812
event_time_utc_ts           28.712871
date_of_birth_2             22.772277
year_of_birth_1             22.772277
last_name                    0.000000
last_modified_ts             0.000000
nickname                     0.000000
group2                       0.000000
phone_number                 0.000000
group1                       0.000000
first_name  

In [408]:
user_attributes_df['ethnicity'].value_counts()

ethnicity
[{'S': 'white'}]                                        69
[{'S': 'black_or_african_american'}]                    24
[{'S': 'asian'}]                                        17
[{'S': 'hispanic_or_latino'}]                            8
[{'S': 'decline'}]                                       6
[{'S': 'hispanic_or_latino'}, {'S': 'white'}]            5
[{'S': 'white'}, {'S': 'hispanic_or_latino'}]            2
[{'S': 'white'}, {'S': 'black_or_african_american'}]     2
[{'S': 'unknown'}]                                       2
[{'S': 'asian'}, {'S': 'white'}]                         2
Name: count, dtype: int64

In [409]:
# Define a function to extract ethnicity
# def extract_preferences(preferences):
#     pref_dict = {}
#     if isinstance(preferences, list):
#         for pref in preferences:
#             key = pref['S']
#             if key == 'decline':
#                 key = 'ethnicity_decline'
#             pref_dict[key] = 1
#             #pref_dict[pref['S']] = 1
#     return pd.Series(pref_dict).fillna(0).astype(int)

# # Apply the function to the 'communication_preference' column
# preferences_df = user_attributes_df['ethnicity'].apply(extract_preferences)
# preferences_df = preferences_df.fillna(0).astype(int)
                                                 
# # Concatenate the new columns with the original DataFrame
# user_attributes_df = pd.concat([user_attributes_df, preferences_df], axis=1)
# user_attributes_df = user_attributes_df.loc[:, ~user_attributes_df.columns.duplicated()]


# user_attributes_df.head()

In [410]:
user_attributes_df.columns

Index(['clinician_provider_id', 'communication_preference', 'date_of_birth_1',
       'date_of_birth_2', 'deactivation_status', 'deactivation_ts',
       'deactivation_user', 'deactivation_user_name', 'dt_complete',
       'ethnicity', 'event_time_utc_ts', 'first_name', 'gender_identity',
       'group1', 'group2', 'last_modified_by', 'last_modified_ts', 'last_name',
       'living_situation', 'nickname', 'notes', 'phone_number', 'phone_usage',
       'sex_at_birth', 'sexual_orientation', 'time_away_from_home', 'user_id',
       'user_type', 'version', 'year_of_birth_1', 'year_of_birth_2'],
      dtype='object')

In [411]:
user_attributes_df = user_attributes_df[['user_id', 'gender_identity', 'sex_at_birth', 'sexual_orientation', 'ethnicity', 
       #                                   'asian',
       # 'white', 'black_or_african_american', 'hispanic_or_latino',
       # 'ethnicity_decline', 'unknown'
       ]]

user_attributes_df.head()

,user_id,gender_identity,sex_at_birth,sexual_orientation,ethnicity
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]


In [412]:
user_attributes_df.ethnicity.value_counts()

ethnicity
[{'S': 'white'}]                                        69
[{'S': 'black_or_african_american'}]                    24
[{'S': 'asian'}]                                        17
[{'S': 'hispanic_or_latino'}]                            8
[{'S': 'decline'}]                                       6
[{'S': 'hispanic_or_latino'}, {'S': 'white'}]            5
[{'S': 'white'}, {'S': 'hispanic_or_latino'}]            2
[{'S': 'white'}, {'S': 'black_or_african_american'}]     2
[{'S': 'unknown'}]                                       2
[{'S': 'asian'}, {'S': 'white'}]                         2
Name: count, dtype: int64

In [413]:
user_attributes_df.head()

,user_id,gender_identity,sex_at_birth,sexual_orientation,ethnicity
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]


In [414]:
user_attributes_df.shape

(404, 5)

# Self Report

In [415]:
folder_path = r'C:\Users\programming.com\Downloads\dtc_SelfReport'

# Get the list of all files in the folder
file_list = os.listdir(folder_path)

# Print the names of the files
files = [file_name for file_name in file_list]
files

['bf3zqijwfy3onplwryhwbc3fuq.json.gz',
 'cmku4kohl45g3n23wmyumrn2pu.json.gz',
 'tsyrqjcb7u5mlmnm5hlnjopoqy.json.gz',
 'x73nm6hhsqyzpbhttes75qurpq.json.gz']

In [416]:
# Initialize an empty list to store DataFrames
dataframes = []
file_paths = []
# Loop through each file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.json.gz'):
        file_path = os.path.join(folder_path, file_name)
        file_paths.append(file_path)
        

# Function to flatten the nested structure in the 'Item' column
def flatten_item_column(df):
    # Ensure the 'Item' column is converted to a DataFrame
    if 'Item' in df.columns:
        expanded = pd.json_normalize(df['Item'])  # Flatten the nested JSON in 'Item'
        return pd.concat([df.drop(columns=['Item']), expanded], axis=1)
    return df

# Loop through each file path and process the .json.gz files
for file_path in file_paths:
    try:
        # Read the JSON file into a DataFrame
        df = pd.read_json(file_path, compression='gzip', lines=True)
        
        # Flatten the 'Item' column if it exists
        df = flatten_item_column(df)
        
        dataframes.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

In [417]:
# Combine all DataFrames into a single DataFrame
self_report_df = pd.concat(dataframes, ignore_index=True)

# Display the combined DataFrame
self_report_df.head()

,user_id.S,timestamp.S,event_type.S,processing_time.S,scores.L,task_identifier.S,json.M.response.L,json.M.task_identifier.S,json.M.survey_name.S,batch_id.S,id.S,type.N,survey_name.S
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:06.554000-05:00,SelfReport,2024-11-25T23:16:58.181031+00:00,"[{'M': {'val': {'N': '7'}, 'q_id': {'S': 'qual...",quality_of_life,[{'M': {'end_date': {'S': '2024-11-25T18:16:50...,quality_of_life,quality_of_life,98336D7B-6CCD-4668-B872-876F4C213A58,caa61329-8456-4b6b-aebd-f99e3a11057c,7,quality_of_life
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:55.116000-05:00,SelfReport,2024-11-25T23:17:34.099013+00:00,"[{'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7...",gad7,[{'M': {'end_date': {'S': '2024-11-25T18:16:59...,gad7,gad7,1EEABBD9-1359-4F58-BE14-2CD60FCB033C,f616c0ee-6e80-4aef-b1ac-d62db1a4d7c3,7,gad7
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:17:34.732000-05:00,SelfReport,2024-12-03T01:37:28.413600+00:00,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'gene...",mental_health_history,[{'M': {'end_date': {'S': '2024-12-02T20:37:26...,mental_health_history,mental_health_history,EDC25961-43FD-4D63-A9BE-1BBDA591C970,e5f2c097-57dd-49fe-919b-ef6f363d664d,7,mental_health_history
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,SelfReport,2024-12-03T01:36:39.818931+00:00,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'phq8...",phq9_with_outro,[{'M': {'end_date': {'S': '2024-12-02T20:36:09...,phq9_with_outro,phq9_with_outro,98CE3A50-95BE-48E7-AE5E-C2356360A854,5b7ee23f-9b92-4fd8-b15c-bd9510efc64b,7,phq9_with_outro
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:34:48.803000-05:00,SelfReport,2024-12-03T01:35:30.449414+00:00,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'qual...",quality_of_life,[{'M': {'end_date': {'S': '2024-12-02T20:34:52...,quality_of_life,quality_of_life,3988E4DF-B614-4FAC-8179-03D279DD8ADB,41a06bc6-1292-499e-a10e-a16e6c9cebaa,7,quality_of_life


In [418]:
self_report_df.shape

(674, 13)

In [419]:
# Clean column names by removing everything after the first dot
def clean_column_names(df):
    df.columns = [col.split('.')[0] for col in df.columns]  # Remove everything after the first dot
    return df

# Apply this cleaning function after combining the DataFrames
self_report_df = pd.concat(dataframes, ignore_index=True).sort_index(axis=1, ascending=True)
self_report_df = clean_column_names(self_report_df)

self_report_df.head(3)

,batch_id,event_type,id,json,json,json,processing_time,scores,survey_name,task_identifier,timestamp,type,user_id
0,98336D7B-6CCD-4668-B872-876F4C213A58,SelfReport,caa61329-8456-4b6b-aebd-f99e3a11057c,[{'M': {'end_date': {'S': '2024-11-25T18:16:50...,quality_of_life,quality_of_life,2024-11-25T23:16:58.181031+00:00,"[{'M': {'val': {'N': '7'}, 'q_id': {'S': 'qual...",quality_of_life,quality_of_life,2024-11-25T18:16:06.554000-05:00,7,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
1,1EEABBD9-1359-4F58-BE14-2CD60FCB033C,SelfReport,f616c0ee-6e80-4aef-b1ac-d62db1a4d7c3,[{'M': {'end_date': {'S': '2024-11-25T18:16:59...,gad7,gad7,2024-11-25T23:17:34.099013+00:00,"[{'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7...",gad7,gad7,2024-11-25T18:16:55.116000-05:00,7,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
2,EDC25961-43FD-4D63-A9BE-1BBDA591C970,SelfReport,e5f2c097-57dd-49fe-919b-ef6f363d664d,[{'M': {'end_date': {'S': '2024-12-02T20:37:26...,mental_health_history,mental_health_history,2024-12-03T01:37:28.413600+00:00,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'gene...",mental_health_history,mental_health_history,2024-11-25T18:17:34.732000-05:00,7,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654


In [420]:
self_report_df.shape


(674, 13)

In [421]:
self_report_df.columns

Index(['batch_id', 'event_type', 'id', 'json', 'json', 'json',
       'processing_time', 'scores', 'survey_name', 'task_identifier',
       'timestamp', 'type', 'user_id'],
      dtype='object')

In [422]:
self_report_df['json']

,json,json,json
0,[{'M': {'end_date': {'S': '2024-11-25T18:16:50...,quality_of_life,quality_of_life
1,[{'M': {'end_date': {'S': '2024-11-25T18:16:59...,gad7,gad7
2,[{'M': {'end_date': {'S': '2024-12-02T20:37:26...,mental_health_history,mental_health_history
3,[{'M': {'end_date': {'S': '2024-12-02T20:36:09...,phq9_with_outro,phq9_with_outro
4,[{'M': {'end_date': {'S': '2024-12-02T20:34:52...,quality_of_life,quality_of_life
...,...,...,...
669,[{'M': {'end_date': {'S': '2024-11-07T20:13:40...,phq9_with_outro,phq9_with_outro
670,[{'M': {'end_date': {'S': '2024-11-07T20:15:45...,gad7,gad7
671,[{'M': {'end_date': {'S': '2024-11-08T16:22:18...,quality_of_life,quality_of_life
672,[{'M': {'end_date': {'S': '2024-11-11T17:34:28...,quality_of_life,quality_of_life


In [423]:
# Get the list of columns
columns = self_report_df.columns

# Rename the columns with similar names 'json'
new_columns = []
json_count = 1
for col in columns:
    if 'json' in col:
        new_columns.append(f'raw_{json_count}')
        json_count += 1
    else:
        new_columns.append(col)

# Assign the new column names to the DataFrame
self_report_df.columns = new_columns

self_report_df.drop(columns=['batch_id', 'type',	'event_type',	'id',	'raw_2',	'raw_3',	'processing_time', 'task_identifier'], inplace=True)

    
# Display the DataFrame with renamed columns

self_report_df.head()

,raw_1,scores,survey_name,timestamp,user_id
0,[{'M': {'end_date': {'S': '2024-11-25T18:16:50...,"[{'M': {'val': {'N': '7'}, 'q_id': {'S': 'qual...",quality_of_life,2024-11-25T18:16:06.554000-05:00,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
1,[{'M': {'end_date': {'S': '2024-11-25T18:16:59...,"[{'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7...",gad7,2024-11-25T18:16:55.116000-05:00,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
2,[{'M': {'end_date': {'S': '2024-12-02T20:37:26...,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'gene...",mental_health_history,2024-11-25T18:17:34.732000-05:00,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
3,[{'M': {'end_date': {'S': '2024-12-02T20:36:09...,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'phq8...",phq9_with_outro,2024-12-02T20:33:56.380000-05:00,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
4,[{'M': {'end_date': {'S': '2024-12-02T20:34:52...,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'qual...",quality_of_life,2024-12-02T20:34:48.803000-05:00,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654


In [424]:
#write a python code to arrange the Dataframe based on the below coloumns in below order.
#user_id, timestamp, survey_name, raw_1, scores, survey_name
self_report_df = self_report_df[['user_id', 'timestamp', 'survey_name', 'raw_1', 'scores', 'survey_name']]
self_report_df.head()


,user_id,timestamp,survey_name,raw_1,scores,survey_name
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:06.554000-05:00,quality_of_life,[{'M': {'end_date': {'S': '2024-11-25T18:16:50...,"[{'M': {'val': {'N': '7'}, 'q_id': {'S': 'qual...",quality_of_life
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:55.116000-05:00,gad7,[{'M': {'end_date': {'S': '2024-11-25T18:16:59...,"[{'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7...",gad7
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:17:34.732000-05:00,mental_health_history,[{'M': {'end_date': {'S': '2024-12-02T20:37:26...,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'gene...",mental_health_history
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,[{'M': {'end_date': {'S': '2024-12-02T20:36:09...,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'phq8...",phq9_with_outro
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:34:48.803000-05:00,quality_of_life,[{'M': {'end_date': {'S': '2024-12-02T20:34:52...,"[{'M': {'val': {'N': '0'}, 'q_id': {'S': 'qual...",quality_of_life


In [425]:
self_report_df.loc[4, 'raw_1']

[{'M': {'end_date': {'S': '2024-12-02T20:34:52.614000-05:00'},
   'identifier': {'S': 'quality_of_life_intro'},
   'type': {'S': 'instruction'},
   'start_date': {'S': '2024-12-02T20:34:48.804000-05:00'}}},
 {'M': {'end_date': {'S': '2024-12-02T20:35:05.967000-05:00'},
   'identifier': {'S': 'quality_of_life_q3'},
   'answer': {'L': [{'S': '2'}]},
   'type': {'S': 'singleChoiceText'},
   'start_date': {'S': '2024-12-02T20:35:01.117000-05:00'}}},
 {'M': {'end_date': {'S': '2024-12-02T20:35:13.219000-05:00'},
   'identifier': {'S': 'quality_of_life_q5'},
   'answer': {'L': [{'S': '2'}]},
   'type': {'S': 'singleChoiceText'},
   'start_date': {'S': '2024-12-02T20:35:09.452000-05:00'}}},
 {'M': {'end_date': {'S': '2024-12-02T20:35:09.451000-05:00'},
   'identifier': {'S': 'quality_of_life_q4'},
   'answer': {'L': [{'S': '2'}]},
   'type': {'S': 'singleChoiceText'},
   'start_date': {'S': '2024-12-02T20:35:05.968000-05:00'}}},
 {'M': {'end_date': {'S': '2024-12-02T20:35:19.688000-05:00'},
 

In [426]:
self_report_df.loc[1, 'scores']

[{'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7_q1'}}},
 {'M': {'val': {'N': '0'}, 'q_id': {'S': 'gad7_q5'}}},
 {'M': {'val': {'N': '0'}, 'q_id': {'S': 'generic_outro'}}},
 {'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7_q2'}}},
 {'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7_q4'}}},
 {'M': {'val': {'N': '0'}, 'q_id': {'S': 'gad7_intro'}}},
 {'M': {'val': {'N': '1'}, 'q_id': {'S': 'gad7_q6'}}},
 {'M': {'val': {'N': '0'}, 'q_id': {'S': 'gad7_q7'}}},
 {'M': {'val': {'N': '0'}, 'q_id': {'S': 'gad7_q3'}}},
 {'M': {'val': {'N': '4'}, 'q_id': {'S': 'gad7_total'}}}]

In [427]:
import pandas as pd

# Assuming 'self_report_df' is your DataFrame
# Extract the 'raw_1' column
raw_1 = self_report_df.loc[4, 'raw_1']

# Function to extract values from the nested structure
def extract_raw_values(raw_1):
    extracted_values = []
    for entry in raw_1:
        end_date = entry['M'].get('end_date', {}).get('S', None)
        identifier = entry['M'].get('identifier', {}).get('S', None)
        answer = entry['M'].get('answer', {}).get('L', [{}])[0].get('S', None)
        type_ = entry['M'].get('type', {}).get('S', None)
        start_date = entry['M'].get('start_date', {}).get('S', None)
        extracted_values.append({
            'end_date': end_date,
            'identifier': identifier,
            'answer': answer,
            'type': type_,
            'start_date': start_date
        })
    return extracted_values

# Extract the values
extracted_values = extract_raw_values(raw_1)

# Display the extracted values
extracted_values

[{'end_date': '2024-12-02T20:34:52.614000-05:00',
  'identifier': 'quality_of_life_intro',
  'answer': None,
  'type': 'instruction',
  'start_date': '2024-12-02T20:34:48.804000-05:00'},
 {'end_date': '2024-12-02T20:35:05.967000-05:00',
  'identifier': 'quality_of_life_q3',
  'answer': '2',
  'type': 'singleChoiceText',
  'start_date': '2024-12-02T20:35:01.117000-05:00'},
 {'end_date': '2024-12-02T20:35:13.219000-05:00',
  'identifier': 'quality_of_life_q5',
  'answer': '2',
  'type': 'singleChoiceText',
  'start_date': '2024-12-02T20:35:09.452000-05:00'},
 {'end_date': '2024-12-02T20:35:09.451000-05:00',
  'identifier': 'quality_of_life_q4',
  'answer': '2',
  'type': 'singleChoiceText',
  'start_date': '2024-12-02T20:35:05.968000-05:00'},
 {'end_date': '2024-12-02T20:35:19.688000-05:00',
  'identifier': 'quality_of_life_q7',
  'answer': '2',
  'type': 'singleChoiceText',
  'start_date': '2024-12-02T20:35:16.771000-05:00'},
 {'end_date': '2024-12-02T20:35:25.139000-05:00',
  'identifi

In [428]:
import pandas as pd

# Assuming 'self_report_df' is your DataFrame
# Extract the 'scores' column
scores = self_report_df.loc[1, 'scores']

# Function to extract values from the nested structure
def extract_scores(scores):
    extracted_scores = []
    for entry in scores:
        val = entry['M']['val']['N']
        q_id = entry['M']['q_id']['S']
        extracted_scores.append({'q_id': q_id, 'val': val})
    return extracted_scores

# Extract the scores
extracted_scores = extract_scores(scores)

# Display the extracted scres
extracted_scores


[{'q_id': 'gad7_q1', 'val': '1'},
 {'q_id': 'gad7_q5', 'val': '0'},
 {'q_id': 'generic_outro', 'val': '0'},
 {'q_id': 'gad7_q2', 'val': '1'},
 {'q_id': 'gad7_q4', 'val': '1'},
 {'q_id': 'gad7_intro', 'val': '0'},
 {'q_id': 'gad7_q6', 'val': '1'},
 {'q_id': 'gad7_q7', 'val': '0'},
 {'q_id': 'gad7_q3', 'val': '0'},
 {'q_id': 'gad7_total', 'val': '4'}]

In [429]:
def extract_scores(scores):
    extracted_scores = {}
    for entry in scores:
        q_id = entry['M']['q_id']['S']
        val = entry['M']['val']['N']
        extracted_scores[q_id] = val
    return extracted_scores
extract_scores(self_report_df['scores'].tolist()[1])

{'gad7_q1': '1',
 'gad7_q5': '0',
 'generic_outro': '0',
 'gad7_q2': '1',
 'gad7_q4': '1',
 'gad7_intro': '0',
 'gad7_q6': '1',
 'gad7_q7': '0',
 'gad7_q3': '0',
 'gad7_total': '4'}

In [430]:
scores = self_report_df['scores']

# Function to extract values from the nested structure and create a dictionary
def extract_scores(scores):
    extracted_scores = {}
    for entry in scores:
        q_id = entry['M']['q_id']['S']
        val = entry['M']['val'].get('N', None)  # Use .get() to handle missing 'N' key
        extracted_scores[q_id] = val
    return extracted_scores

# Apply the function to each row in the 'scores' column and create a new DataFrame
extracted_scores_df = pd.DataFrame(scores.apply(extract_scores).tolist())

# Merge the extracted scores DataFrame with the original DataFrame
self_report_df = pd.concat([self_report_df, extracted_scores_df], axis=1)

# Drop the original 'scores' column if no longer needed
self_report_df.drop(columns=['raw_1',	'scores'], inplace=True)


In [431]:
self_report_df.columns

Index(['user_id', 'timestamp', 'survey_name', 'survey_name',
       'quality_of_life_q8', 'generic_outro', 'quality_of_life_q1',
       'quality_of_life_q4', 'quality_of_life_q6', 'quality_of_life_q2',
       'quality_of_life_q7', 'quality_of_life_intro', 'quality_of_life_q3',
       'quality_of_life_q5', 'quality_of_life_total', 'gad7_q1', 'gad7_q5',
       'gad7_q2', 'gad7_q4', 'gad7_intro', 'gad7_q6', 'gad7_q7', 'gad7_q3',
       'gad7_total', 'mental_health_history_intro', 'mental_health_history_q3',
       'mental_health_history_q1', 'mental_health_history_q7',
       'mental_health_history_q2', 'mental_health_history_q6',
       'mental_health_history_q5', 'mental_health_history_q9',
       'mental_health_history_q8', 'phq8_q5', 'phq8_q7', 'phq8_q4',
       'phq9_intro', 'phq8_q1', 'phq8_q2', 'phq8_q8', 'phq8_q6', 'phq8_q3',
       'phq9_outro', 'phq9_q9', 'phq9_with_outro_total', 'phq_total',
       'phq_9_with_outro_total', 'mental_health_history_q4',
       'invalid_format_qua

In [432]:
self_report_df.survey_name

,survey_name,survey_name
0,quality_of_life,quality_of_life
1,gad7,gad7
2,mental_health_history,mental_health_history
3,phq9_with_outro,phq9_with_outro
4,quality_of_life,quality_of_life
...,...,...
669,phq9_with_outro,phq9_with_outro
670,gad7,gad7
671,quality_of_life,quality_of_life
672,quality_of_life,quality_of_life


In [433]:

# Display the transformed DataFrame
self_report_df.head()

,user_id,timestamp,survey_name,survey_name,quality_of_life_q8,generic_outro,quality_of_life_q1,quality_of_life_q4,quality_of_life_q6,quality_of_life_q2,...,phq8_q8,phq8_q6,phq8_q3,phq9_outro,phq9_q9,phq9_with_outro_total,phq_total,phq_9_with_outro_total,mental_health_history_q4,invalid_format_quality_of_life
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:06.554000-05:00,quality_of_life,quality_of_life,7,0,2,2,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:55.116000-05:00,gad7,gad7,NaN,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:17:34.732000-05:00,mental_health_history,mental_health_history,NaN,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,5,5,5,NaN,NaN
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:34:48.803000-05:00,quality_of_life,quality_of_life,7,0,2,2,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [434]:
self_report_df.shape

(674, 49)

In [435]:
self_report_df.columns

Index(['user_id', 'timestamp', 'survey_name', 'survey_name',
       'quality_of_life_q8', 'generic_outro', 'quality_of_life_q1',
       'quality_of_life_q4', 'quality_of_life_q6', 'quality_of_life_q2',
       'quality_of_life_q7', 'quality_of_life_intro', 'quality_of_life_q3',
       'quality_of_life_q5', 'quality_of_life_total', 'gad7_q1', 'gad7_q5',
       'gad7_q2', 'gad7_q4', 'gad7_intro', 'gad7_q6', 'gad7_q7', 'gad7_q3',
       'gad7_total', 'mental_health_history_intro', 'mental_health_history_q3',
       'mental_health_history_q1', 'mental_health_history_q7',
       'mental_health_history_q2', 'mental_health_history_q6',
       'mental_health_history_q5', 'mental_health_history_q9',
       'mental_health_history_q8', 'phq8_q5', 'phq8_q7', 'phq8_q4',
       'phq9_intro', 'phq8_q1', 'phq8_q2', 'phq8_q8', 'phq8_q6', 'phq8_q3',
       'phq9_outro', 'phq9_q9', 'phq9_with_outro_total', 'phq_total',
       'phq_9_with_outro_total', 'mental_health_history_q4',
       'invalid_format_qua

In [436]:
self_report_df = self_report_df.loc[:, ~self_report_df.columns.duplicated()]

self_report_df.head()


,user_id,timestamp,survey_name,quality_of_life_q8,generic_outro,quality_of_life_q1,quality_of_life_q4,quality_of_life_q6,quality_of_life_q2,quality_of_life_q7,...,phq8_q8,phq8_q6,phq8_q3,phq9_outro,phq9_q9,phq9_with_outro_total,phq_total,phq_9_with_outro_total,mental_health_history_q4,invalid_format_quality_of_life
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:06.554000-05:00,quality_of_life,7,0,2,2,2,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:16:55.116000-05:00,gad7,NaN,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-11-25T18:17:34.732000-05:00,mental_health_history,NaN,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,5,5,5,NaN,NaN
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:34:48.803000-05:00,quality_of_life,7,0,2,2,2,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [437]:

self_report_df.shape

(674, 48)

In [438]:
self_report_df[self_report_df['user_id'] == 'us-east-1:db841a21-c44c-cae0-4d10-3576737ed654'].loc[1]


user_id                           us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
timestamp                                       2024-11-25T18:16:55.116000-05:00
survey_name                                                                 gad7
quality_of_life_q8                                                           NaN
generic_outro                                                                  0
quality_of_life_q1                                                           NaN
quality_of_life_q4                                                           NaN
quality_of_life_q6                                                           NaN
quality_of_life_q2                                                           NaN
quality_of_life_q7                                                           NaN
quality_of_life_intro                                                        NaN
quality_of_life_q3                                                           NaN
quality_of_life_q5          

In [439]:
self_report_df.columns

Index(['user_id', 'timestamp', 'survey_name', 'quality_of_life_q8',
       'generic_outro', 'quality_of_life_q1', 'quality_of_life_q4',
       'quality_of_life_q6', 'quality_of_life_q2', 'quality_of_life_q7',
       'quality_of_life_intro', 'quality_of_life_q3', 'quality_of_life_q5',
       'quality_of_life_total', 'gad7_q1', 'gad7_q5', 'gad7_q2', 'gad7_q4',
       'gad7_intro', 'gad7_q6', 'gad7_q7', 'gad7_q3', 'gad7_total',
       'mental_health_history_intro', 'mental_health_history_q3',
       'mental_health_history_q1', 'mental_health_history_q7',
       'mental_health_history_q2', 'mental_health_history_q6',
       'mental_health_history_q5', 'mental_health_history_q9',
       'mental_health_history_q8', 'phq8_q5', 'phq8_q7', 'phq8_q4',
       'phq9_intro', 'phq8_q1', 'phq8_q2', 'phq8_q8', 'phq8_q6', 'phq8_q3',
       'phq9_outro', 'phq9_q9', 'phq9_with_outro_total', 'phq_total',
       'phq_9_with_outro_total', 'mental_health_history_q4',
       'invalid_format_quality_of_life'],

In [440]:
self_report_df.user_id.shape

(674,)

In [441]:
self_report_df.user_id.nunique()

57

In [442]:
self_report_df = self_report_df[self_report_df.survey_name == 'phq9_with_outro']
self_report_df.head()

,user_id,timestamp,survey_name,quality_of_life_q8,generic_outro,quality_of_life_q1,quality_of_life_q4,quality_of_life_q6,quality_of_life_q2,quality_of_life_q7,...,phq8_q8,phq8_q6,phq8_q3,phq9_outro,phq9_q9,phq9_with_outro_total,phq_total,phq_9_with_outro_total,mental_health_history_q4,invalid_format_quality_of_life
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,5,5,5,NaN,NaN
8,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,3,3,3,NaN,NaN
10,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-23T14:15:04.998000-05:00,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,3,3,3,NaN,NaN
12,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2025-01-06T10:39:47.324000-05:00,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,2,2,2,NaN,NaN
18,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,phq9_with_outro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,2,2,0,0,9,9,9,NaN,NaN


In [443]:
self_report_df.shape

(157, 48)

In [444]:
self_report_df.user_id.nunique()

52

In [445]:
self_report_df= self_report_df[['user_id', 'timestamp', 'survey_name',  
                           'phq8_q1', 'phq8_q2', 'phq8_q3', 'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8', 'phq9_q9', 'phq9_intro', 'phq9_outro', 'phq_total'
                           ]]

self_report_df.head()

,user_id,timestamp,survey_name,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
8,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3
10,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-23T14:15:04.998000-05:00,phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3
12,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2025-01-06T10:39:47.324000-05:00,phq9_with_outro,0,0,0,1,0,0,1,0,0,0,0,2
18,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,phq9_with_outro,1,1,2,1,1,2,1,0,0,0,0,9


In [446]:
self_report_df.shape

(157, 15)

In [447]:
self_report_df[self_report_df['user_id'] == 'us-east-1:db841a21-c44c-cae0-4d10-3576737ed654'].loc[3]

user_id        us-east-1:db841a21-c44c-cae0-4d10-3576737ed654
timestamp                    2024-12-02T20:33:56.380000-05:00
survey_name                                   phq9_with_outro
phq8_q1                                                     1
phq8_q2                                                     1
phq8_q3                                                     0
phq8_q4                                                     2
phq8_q5                                                     0
phq8_q6                                                     1
phq8_q7                                                     0
phq8_q8                                                     0
phq9_q9                                                     0
phq9_intro                                                  0
phq9_outro                                                  0
phq_total                                                   5
Name: 3, dtype: object

In [448]:
self_report_df.user_id.nunique()

52

In [449]:
print('Shape of the self_report_df table: ', self_report_df.shape)
print('Unique users from self_report_df table: ', self_report_df.user_id.nunique())

Shape of the self_report_df table:  (157, 15)
Unique users from self_report_df table:  52


# Merge Self Report and User attributes Of DTC Users

In [450]:
print('Shape of the user attributes table: ', user_attributes_df.shape)
print('Unique users from user attributes table: ', user_attributes_df.user_id.nunique())
user_attributes_df.head()

Shape of the user attributes table:  (404, 5)
Unique users from user attributes table:  116


,user_id,gender_identity,sex_at_birth,sexual_orientation,ethnicity
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]


In [451]:
user_attributes_df.columns

Index(['user_id', 'gender_identity', 'sex_at_birth', 'sexual_orientation',
       'ethnicity'],
      dtype='object')

In [452]:
self_report_df.columns

Index(['user_id', 'timestamp', 'survey_name', 'phq8_q1', 'phq8_q2', 'phq8_q3',
       'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8', 'phq9_q9',
       'phq9_intro', 'phq9_outro', 'phq_total'],
      dtype='object')

In [453]:

# 1. List unique user_ids from each table:
unique_user_ids_user_attributes = user_attributes_df['user_id'].unique()
unique_user_ids_self_report = self_report_df['user_id'].unique()

# Optional: Find out which user_ids are in user_attributes_df but not in self_report_df
missing_in_self_report = set(unique_user_ids_user_attributes) - set(unique_user_ids_self_report)
missing_in_self_report

{'us-east-1:db841a21-c403-c4ba-ebd1-e5729c102f85',
 'us-east-1:db841a21-c405-cadd-8ecd-23a509c7f4f4',
 'us-east-1:db841a21-c40b-c29a-17b4-574c7824f3cc',
 'us-east-1:db841a21-c420-c16b-88a0-f4c6b55dc46f',
 'us-east-1:db841a21-c428-ce1f-16fa-0de4f3e477f2',
 'us-east-1:db841a21-c42b-ca48-2634-02a92db6d920',
 'us-east-1:db841a21-c431-c58b-efbb-3d2c6842ad43',
 'us-east-1:db841a21-c432-cb9e-e738-48a723192af3',
 'us-east-1:db841a21-c434-c0b0-c35b-caee729ab67c',
 'us-east-1:db841a21-c437-c442-0e55-c081921df95e',
 'us-east-1:db841a21-c437-ce1b-5feb-ed2576ee5788',
 'us-east-1:db841a21-c438-c765-5cd2-cc0bea1bc0b0',
 'us-east-1:db841a21-c43b-ce1d-bfe8-eae02f6ce9f1',
 'us-east-1:db841a21-c43c-c330-2a72-fab1ba257ed4',
 'us-east-1:db841a21-c43c-cc24-a71b-742d92dae983',
 'us-east-1:db841a21-c442-c963-ef3d-308654eb350e',
 'us-east-1:db841a21-c443-c774-01a0-520a1cc9c292',
 'us-east-1:db841a21-c444-cbc6-7c77-87ae4dd9cbe1',
 'us-east-1:db841a21-c444-ccd8-abb7-c8cb5cbc52d9',
 'us-east-1:db841a21-c447-cb27-

In [454]:
# missing_in_self_report is defined as a set of user IDs that were found missing in the self-report table.

filtered_user_attributes_df = user_attributes_df[user_attributes_df['user_id'].isin(missing_in_self_report)]
filtered_user_attributes_df


,user_id,gender_identity,sex_at_birth,sexual_orientation,ethnicity
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,male,male,straight_or_heterosexual,[{'S': 'asian'}]
...,...,...,...,...,...
399,us-east-1:db841a21-c42b-ca48-2634-02a92db6d920,NaN,NaN,NaN,NaN
400,us-east-1:db841a21-c42b-ca48-2634-02a92db6d920,NaN,NaN,NaN,NaN
401,us-east-1:db841a21-c4f3-c260-a350-3e9e8ad7c935,NaN,NaN,NaN,NaN
402,us-east-1:db841a21-c4f3-c260-a350-3e9e8ad7c935,NaN,NaN,NaN,NaN


In [455]:

# 2. Join the two tables on 'user_id'
# Using an inner join will keep only those user_ids present in both tables.
merged_df = pd.merge(self_report_df, user_attributes_df, on='user_id', how='inner')
merged_df.head()

,user_id,timestamp,survey_name,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total,gender_identity,sex_at_birth,sexual_orientation,ethnicity
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}]
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}]
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}]


In [456]:
merged_df= merged_df[['user_id', 'timestamp', 
                      'gender_identity', 'sex_at_birth', 'sexual_orientation', 'ethnicity',
       'survey_name', 'phq8_q1', 'phq8_q2', 'phq8_q3', 'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8', 'phq9_q9', 'phq9_intro', 'phq9_outro', 'phq_total',]]

merged_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3


In [457]:
merged_df.shape

(685, 19)

In [458]:
merged_df.user_id.nunique()

52

In [459]:
merged_df.columns

Index(['user_id', 'timestamp', 'gender_identity', 'sex_at_birth',
       'sexual_orientation', 'ethnicity', 'survey_name', 'phq8_q1', 'phq8_q2',
       'phq8_q3', 'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8',
       'phq9_q9', 'phq9_intro', 'phq9_outro', 'phq_total'],
      dtype='object')

In [460]:
merged_df_unique_users_id = list(merged_df.user_id.unique())
merged_df_unique_users_id

['us-east-1:db841a21-c44c-cae0-4d10-3576737ed654',
 'us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1',
 'us-east-1:db841a21-c406-cbe8-8908-c7904aafe112',
 'us-east-1:db841a21-c4d2-ce0f-dbef-f0621daa34e9',
 'us-east-1:db841a21-c471-c321-a10d-3679fbf43562',
 'us-east-1:db841a21-c4aa-c659-c0e0-b1385645b311',
 'us-east-1:db841a21-c468-cd6d-f7ab-edb30daa7e62',
 'us-east-1:db841a21-c4fc-c053-ff60-9e408e777bd3',
 'us-east-1:db841a21-c45b-cbcc-6e86-28bf1f056f0e',
 'us-east-1:db841a21-c472-c904-9736-4281ee48e8a7',
 'us-east-1:db841a21-c4cf-c0f9-210b-e311de6b881b',
 'us-east-1:db841a21-c4cf-c1ad-b90a-78145dd7ca9c',
 'us-east-1:db841a21-c45f-cdc9-e7ac-ccae31af5676',
 'us-east-1:db841a21-c4ce-c0c6-34fd-af71468f2c40',
 'us-east-1:db841a21-c428-cd84-e285-398cf452c217',
 'us-east-1:db841a21-c4a7-ced2-1395-c814829d5426',
 'us-east-1:db841a21-c4b6-c79a-ef9c-3cd3c42019a4',
 'us-east-1:db841a21-c4c3-c1cf-6500-e8736fa04321',
 'us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2',
 'us-east-1:db841a21-c4ce-cb7d-

# Location Dataset

In [461]:
folder_path = r'C:\Users\programming.com\Downloads\dtc_Location'

# Get the list of all files in the folder
file_list = os.listdir(folder_path)

# Print the names of the files
files = [file_name for file_name in file_list]
files

['ndhdn7zlvi7q3lw5adlrszjb6e.json.gz',
 'wqjndrrj6ayvfagp6x7xpuhz4u.json.gz',
 'xf3aznqnya7mbcm7lircfi5d5a.json.gz',
 'xlmkco6px42e3en5ms46pgppji.json.gz']

In [462]:
# Initialize an empty list to store DataFrames
dataframes = []
file_paths = []
# Loop through each file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.json.gz'):
        file_path = os.path.join(folder_path, file_name)
        file_paths.append(file_path)
        

# Function to flatten the nested structure in the 'Item' column
def flatten_item_column(df):
    # Ensure the 'Item' column is converted to a DataFrame
    if 'Item' in df.columns:
        expanded = pd.json_normalize(df['Item'])  # Flatten the nested JSON in 'Item'
        return pd.concat([df.drop(columns=['Item']), expanded], axis=1)
    return df

# Loop through each file path and process the .json.gz files
for file_path in file_paths:
    try:
        # Read the JSON file into a DataFrame
        df = pd.read_json(file_path, compression='gzip', lines=True)
        
        # Flatten the 'Item' column if it exists
        df = flatten_item_column(df)
        
        dataframes.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
# Combine all DataFrames into a single DataFrame
location_df = pd.concat(dataframes, ignore_index=True)

# Display the combined DataFrame
location_df.head()

,user_id.S,timestamp.S,version.N,altitude.N,batch_id.S,accuracy.N,event_type.S,processing_time.S,course.N,longitude.N,speed.N,id.S,latitude.N,duration.N,type.N,horizontal_accuracy.N,vertical_accuracy.N,creation_timestamp.S,init_date.S
0,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:13:19.074000-04:00,2024071587,204.1000061035,6d29f455-cf08-4d12-901c-9ec871828875,0,Location,2024-10-20T21:26:00.879316+00:00,0,-81.1874342,0,a679a080-c3b0-4251-a96b-0e7eab1c96e0,35.2552824,301.881,2,NaN,NaN,NaN,NaN
1,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:18:20.955000-04:00,2024071587,204.3000030518,6d29f455-cf08-4d12-901c-9ec871828875,0,Location,2024-10-20T21:26:00.879316+00:00,0,-81.1873805,0,24cd8aac-0d70-4483-b418-f9b14fd88193,35.2553879,302.26,2,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:23:23.215000-04:00,2024071587,204.3000030518,6d29f455-cf08-4d12-901c-9ec871828875,0,Location,2024-10-20T21:26:00.879316+00:00,0,-81.1873805,0,48129a56-61e1-4eea-888a-1b620a98814a,35.2553879,0.321,2,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:23:23.536000-04:00,2024071587,204.3000030518,4ad149aa-1ce5-43bc-93ce-57d6ea6308bf,0,Location,2024-10-20T21:48:40.031554+00:00,0,-81.187374,0,03e361dc-bfca-45ca-9ad0-19724e4eb94f,35.2553885,264.695,2,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:27:48.231000-04:00,2024071587,204.3000030518,4ad149aa-1ce5-43bc-93ce-57d6ea6308bf,0,Location,2024-10-20T21:48:40.031554+00:00,0,-81.187374,0,96ee42f0-7297-4a9c-82da-73ae12c88c99,35.2553885,340.206,2,NaN,NaN,NaN,NaN


In [463]:
location_df.shape

(141955, 19)

In [464]:
# Clean column names by removing everything after the first dot
def clean_column_names(df):
    df.columns = [col.split('.')[0] for col in df.columns]  # Remove everything after the first dot
    return df

# Apply this cleaning function after combining the DataFrames
location_df = pd.concat(dataframes, ignore_index=True).sort_index(axis=1, ascending=True)
location_df = clean_column_names(location_df)

location_df.head(3)

,accuracy,altitude,batch_id,course,creation_timestamp,duration,event_type,horizontal_accuracy,id,init_date,latitude,longitude,processing_time,speed,timestamp,type,user_id,version,vertical_accuracy
0,0,204.1000061035,6d29f455-cf08-4d12-901c-9ec871828875,0,NaN,301.881,Location,NaN,a679a080-c3b0-4251-a96b-0e7eab1c96e0,NaN,35.2552824,-81.1874342,2024-10-20T21:26:00.879316+00:00,0,2024-10-20T17:13:19.074000-04:00,2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024071587,NaN
1,0,204.3000030518,6d29f455-cf08-4d12-901c-9ec871828875,0,NaN,302.26,Location,NaN,24cd8aac-0d70-4483-b418-f9b14fd88193,NaN,35.2553879,-81.1873805,2024-10-20T21:26:00.879316+00:00,0,2024-10-20T17:18:20.955000-04:00,2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024071587,NaN
2,0,204.3000030518,6d29f455-cf08-4d12-901c-9ec871828875,0,NaN,0.321,Location,NaN,48129a56-61e1-4eea-888a-1b620a98814a,NaN,35.2553879,-81.1873805,2024-10-20T21:26:00.879316+00:00,0,2024-10-20T17:23:23.215000-04:00,2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024071587,NaN


In [465]:
location_df.columns

Index(['accuracy', 'altitude', 'batch_id', 'course', 'creation_timestamp',
       'duration', 'event_type', 'horizontal_accuracy', 'id', 'init_date',
       'latitude', 'longitude', 'processing_time', 'speed', 'timestamp',
       'type', 'user_id', 'version', 'vertical_accuracy'],
      dtype='object')

In [466]:
location_df= location_df[['user_id', 'timestamp', 'latitude', 'longitude', 'altitude', 
                          #'accuracy', 'speed', 'course', 'horizontal_accuracy', 'vertical_accuracy', 'duration'
                          ]]
location_df.head()

,user_id,timestamp,latitude,longitude,altitude
0,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:13:19.074000-04:00,35.2552824,-81.1874342,204.1000061035
1,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:18:20.955000-04:00,35.2553879,-81.1873805,204.3000030518
2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:23:23.215000-04:00,35.2553879,-81.1873805,204.3000030518
3,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:23:23.536000-04:00,35.2553885,-81.187374,204.3000030518
4,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:27:48.231000-04:00,35.2553885,-81.187374,204.3000030518


In [467]:
location_df.columns

Index(['user_id', 'timestamp', 'latitude', 'longitude', 'altitude'], dtype='object')

In [468]:
location_df.shape

(141955, 5)

# Merging the location dataframe with the user_attributes and self_report dataframe.

In [469]:
# Merge merged_df with location_df based on 'user_id'
# Using an inner join to keep only the rows with matching user_ids in both dataframes.
merged_df_with_location = pd.merge(merged_df, location_df, on='user_id', how='inner')

merged_df_with_location.head()

,user_id,timestamp_x,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,...,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total,timestamp_y,latitude,longitude,altitude
0,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],phq9_with_outro,1,1,2,...,1,0,0,0,0,9,2024-10-20T17:13:19.074000-04:00,35.2552824,-81.1874342,204.1000061035
1,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],phq9_with_outro,1,1,2,...,1,0,0,0,0,9,2024-10-20T17:18:20.955000-04:00,35.2553879,-81.1873805,204.3000030518
2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],phq9_with_outro,1,1,2,...,1,0,0,0,0,9,2024-10-20T17:23:23.215000-04:00,35.2553879,-81.1873805,204.3000030518
3,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],phq9_with_outro,1,1,2,...,1,0,0,0,0,9,2024-10-20T17:23:23.536000-04:00,35.2553885,-81.187374,204.3000030518
4,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],phq9_with_outro,1,1,2,...,1,0,0,0,0,9,2024-10-20T17:27:48.231000-04:00,35.2553885,-81.187374,204.3000030518


In [470]:
merged_df_with_location.shape

(1993726, 23)

In [471]:
merged_df_with_location.user_id.unique()

array(['us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1',
       'us-east-1:db841a21-c406-cbe8-8908-c7904aafe112',
       'us-east-1:db841a21-c4d2-ce0f-dbef-f0621daa34e9',
       'us-east-1:db841a21-c471-c321-a10d-3679fbf43562',
       'us-east-1:db841a21-c4aa-c659-c0e0-b1385645b311',
       'us-east-1:db841a21-c468-cd6d-f7ab-edb30daa7e62',
       'us-east-1:db841a21-c4fc-c053-ff60-9e408e777bd3',
       'us-east-1:db841a21-c45b-cbcc-6e86-28bf1f056f0e',
       'us-east-1:db841a21-c472-c904-9736-4281ee48e8a7',
       'us-east-1:db841a21-c4cf-c1ad-b90a-78145dd7ca9c',
       'us-east-1:db841a21-c45f-cdc9-e7ac-ccae31af5676',
       'us-east-1:db841a21-c4ce-c0c6-34fd-af71468f2c40',
       'us-east-1:db841a21-c428-cd84-e285-398cf452c217',
       'us-east-1:db841a21-c4a7-ced2-1395-c814829d5426',
       'us-east-1:db841a21-c4b6-c79a-ef9c-3cd3c42019a4',
       'us-east-1:db841a21-c4c3-c1cf-6500-e8736fa04321',
       'us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2',
       'us-east-1:db841a21-c4ce

In [472]:
merged_df_with_location.user_id.nunique()

50

In [473]:
merged_df_with_location.columns

Index(['user_id', 'timestamp_x', 'gender_identity', 'sex_at_birth',
       'sexual_orientation', 'ethnicity', 'survey_name', 'phq8_q1', 'phq8_q2',
       'phq8_q3', 'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8',
       'phq9_q9', 'phq9_intro', 'phq9_outro', 'phq_total', 'timestamp_y',
       'latitude', 'longitude', 'altitude'],
      dtype='object')

In [474]:
merged_df_with_location = merged_df_with_location[['user_id', 'timestamp_x', 'timestamp_y', 
                                                   'gender_identity', 'sex_at_birth', 'sexual_orientation', 'ethnicity', 
                                                   'latitude', 'longitude', 'altitude', 
                                                   'survey_name', 'phq8_q1', 'phq8_q2', 'phq8_q3', 'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8', 'phq9_q9', 'phq9_intro', 'phq9_outro', 'phq_total', ]]

merged_df_with_location.head()

,user_id,timestamp_x,timestamp_y,gender_identity,sex_at_birth,sexual_orientation,ethnicity,latitude,longitude,altitude,...,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
0,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:13:19.074000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2552824,-81.1874342,204.1000061035,...,2,1,1,2,1,0,0,0,0,9
1,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:18:20.955000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553879,-81.1873805,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:23:23.215000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553879,-81.1873805,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
3,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:23:23.536000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553885,-81.187374,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
4,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:27:48.231000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553885,-81.187374,204.3000030518,...,2,1,1,2,1,0,0,0,0,9


In [475]:
merged_df_with_location[merged_df_with_location.user_id == 'us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1']

,user_id,timestamp_x,timestamp_y,gender_identity,sex_at_birth,sexual_orientation,ethnicity,latitude,longitude,altitude,...,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
0,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:13:19.074000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2552824,-81.1874342,204.1000061035,...,2,1,1,2,1,0,0,0,0,9
1,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:18:20.955000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553879,-81.1873805,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
2,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:23:23.215000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553879,-81.1873805,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
3,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:23:23.536000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553885,-81.187374,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
4,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-10-20T17:27:48.231000-04:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553885,-81.187374,204.3000030518,...,2,1,1,2,1,0,0,0,0,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16460,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-11-16T00:06:27.950000-05:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2553223,-81.1873591,204.1000061035,...,2,1,1,2,1,0,0,0,0,9
16461,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-11-16T00:11:31.137000-05:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2552892,-81.1873796,204.1000061035,...,2,1,1,2,1,0,0,0,0,9
16462,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-11-16T00:16:33.540000-05:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2552984,-81.1873661,204.1000061035,...,2,1,1,2,1,0,0,0,0,9
16463,us-east-1:db841a21-c4e8-ca31-8042-c2617c66f0a1,2024-10-20T17:16:39.825000-04:00,2024-11-16T00:21:35.452000-05:00,male,female,straight_or_heterosexual,[{'S': 'white'}],35.2552853,-81.1873712,204.1000061035,...,2,1,1,2,1,0,0,0,0,9


In [476]:
merged_df_with_location[merged_df_with_location.user_id == 'us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb']

,user_id,timestamp_x,timestamp_y,gender_identity,sex_at_birth,sexual_orientation,ethnicity,latitude,longitude,altitude,...,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
1940926,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-10-16T13:47:46.524000-04:00,2024-10-18T13:59:36.832000-04:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2393949513,-81.1105921072,238.3902397156,...,2,3,3,1,2,0,1,0,0,16
1940927,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-10-16T13:47:46.524000-04:00,2024-10-18T17:00:53.038000-04:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2440040496,-81.1083171782,226.8937988281,...,2,3,3,1,2,0,1,0,0,16
1940928,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-10-16T13:47:46.524000-04:00,2024-10-18T17:05:53.086000-04:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2533176667,-81.1075008333,217.1,...,2,3,3,1,2,0,1,0,0,16
1940929,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-10-16T13:47:46.524000-04:00,2024-10-18T17:10:53.117000-04:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2663257005,-81.1015026484,237.3953125,...,2,3,3,1,2,0,1,0,0,16
1940930,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-10-16T13:47:46.524000-04:00,2024-10-18T17:14:49.122000-04:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2645185454,-81.0909278831,231.3869779751,...,2,3,3,1,2,0,1,0,0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1984841,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-12-26T14:07:23.109000-05:00,2025-01-15T07:30:25.668000-05:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2612305726,-81.110315149,218.7379617691,...,3,1,3,1,1,0,0,0,0,12
1984842,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-12-26T14:07:23.109000-05:00,2025-01-15T07:35:26.158000-05:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2860007423,-81.1984531902,218.9472471237,...,3,1,3,1,1,0,0,0,0,12
1984843,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-12-26T14:07:23.109000-05:00,2025-01-15T07:40:26.143000-05:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2555740565,-81.2922263231,236.5505145073,...,3,1,3,1,1,0,0,0,0,12
1984844,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,2024-12-26T14:07:23.109000-05:00,2025-01-15T07:45:26.666000-05:00,female,female,straight_or_heterosexual,[{'S': 'white'}],35.2571272993,-81.3839511072,263.0938651085,...,3,1,3,1,1,0,0,0,0,12


# dtc Summary

In [477]:
folder_path = r'C:\Users\programming.com\Downloads\dtc_Summary'

# Get the list of all files in the folder
file_list = os.listdir(folder_path)

# Print the names of the files
files = [file_name for file_name in file_list]
files
# Initialize an empty list to store DataFrames
dataframes = []
file_paths = []
# Loop through each file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.json.gz'):
        file_path = os.path.join(folder_path, file_name)
        file_paths.append(file_path)
        

# Function to flatten the nested structure in the 'Item' column
def flatten_item_column(df):
    # Ensure the 'Item' column is converted to a DataFrame
    if 'Item' in df.columns:
        expanded = pd.json_normalize(df['Item'])  # Flatten the nested JSON in 'Item'
        return pd.concat([df.drop(columns=['Item']), expanded], axis=1)
    return df

# Loop through each file path and process the .json.gz files
for file_path in file_paths:
    try:
        # Read the JSON file into a DataFrame
        df = pd.read_json(file_path, compression='gzip', lines=True)
        
        # Flatten the 'Item' column if it exists
        df = flatten_item_column(df)
        
        dataframes.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
# Combine all DataFrames into a single DataFrame
summary_df = pd.concat(dataframes, ignore_index=True)

# Display the combined DataFrame
summary_df.head()

,user_id.S,date.S,step_floors_down_total.NULL,wsp_efficiency.NULL,total_location_duration.N,activity_walking_count.NULL,burstiness.NULL,self_report_count.N,sp_duration_pedometer.NULL,activity_stationary_mean.NULL,...,summary_daily_7.M.07-17.M.auth_location.N,summary_daily_7.M.07-17.M.ec_SelfReport.N,summary_daily_7.M.07-18.M.coverage.N,summary_daily_7.M.07-18.M.coverage_explanation.S,summary_daily_7.M.07-18.M.auth_notification.N,summary_daily_7.M.07-18.M.auth_activity.N,summary_daily_7.M.07-18.M.auth_explanation.S,summary_daily_7.M.07-18.M.auth_audio.N,summary_daily_7.M.07-18.M.auth_location.N,summary_daily_7.M.07-18.M.ec_SelfReport.N
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-19,True,True,0,True,True,0,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-20,True,True,0,NaN,NaN,0,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-21,True,NaN,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-22,True,NaN,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-23,True,NaN,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [478]:
summary_df.shape

(5879, 2181)

In [479]:
# Clean column names by removing everything after the first dot
def clean_column_names(df):
    df.columns = [col.split('.')[0] for col in df.columns]  # Remove everything after the first dot
    return df

# Apply this cleaning function after combining the DataFrames
summary_df = pd.concat(dataframes, ignore_index=True)
summary_df = clean_column_names(summary_df)

summary_df.head()

,user_id,date,step_floors_down_total,wsp_efficiency,total_location_duration,activity_walking_count,burstiness,self_report_count,sp_duration_pedometer,activity_stationary_mean,...,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7,summary_daily_7
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-19,True,True,0,True,True,0,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-20,True,True,0,NaN,NaN,0,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-21,True,NaN,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-22,True,NaN,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-23,True,NaN,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [480]:
summary_df.shape

(5879, 2181)

In [481]:

duplicate_columns = summary_df.columns[summary_df.columns.duplicated()].unique()

# Iterate through duplicate columns and combine their values
for col in duplicate_columns:
    # Get all columns with the same name
    duplicate_col_set = summary_df.loc[:, summary_df.columns == col]
    
    # Combine them into one column by filling NaN values from left to right
    combined_column = duplicate_col_set.bfill(axis=1).iloc[:, 0]
    
    # Replace the original columns with the combined column
    summary_df[col] = combined_column

# Drop duplicate columns, keeping only one
summary_df = summary_df.loc[:, ~summary_df.columns.duplicated()]

In [482]:
summary_df.shape

(5879, 202)

In [483]:
summary_df.columns

Index(['user_id', 'date', 'step_floors_down_total', 'wsp_efficiency',
       'total_location_duration', 'activity_walking_count', 'burstiness',
       'self_report_count', 'sp_duration_pedometer',
       'activity_stationary_mean',
       ...
       'sleep_start', 'activity_unknown_max', 'activity_walking_mean',
       'loc_entropy', 'last_processed_time', 'daily_time_at_location2_1',
       'daily_time_at_location2_2', 'daily_time_at_location2_5',
       'daily_time_at_location2_3', 'daily_time_at_location2_4'],
      dtype='object', length=202)

In [484]:
summary_df.head()

,user_id,date,step_floors_down_total,wsp_efficiency,total_location_duration,activity_walking_count,burstiness,self_report_count,sp_duration_pedometer,activity_stationary_mean,...,sleep_start,activity_unknown_max,activity_walking_mean,loc_entropy,last_processed_time,daily_time_at_location2_1,daily_time_at_location2_2,daily_time_at_location2_5,daily_time_at_location2_3,daily_time_at_location2_4
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-19,True,True,0,True,True,0,True,True,...,True,True,True,True,2024-07-21T03:06:29.091894,NaN,NaN,NaN,NaN,NaN
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-20,True,True,0,0,0.1182471221,0,True,482.0123888889,...,True,125.58,0,True,2024-07-22T02:12:00.516628,NaN,NaN,NaN,NaN,NaN
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-21,True,1,0,2,0.0685811936,0,25087.481,2373.3392285429,...,2024-07-20T21:50:00.000000,1080.444,61.194,True,2024-07-22T21:52:26.209288,NaN,NaN,NaN,NaN,NaN
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-22,True,6.2692307692,0,0,0.2657709157,0,50939.572,2034.5758292439,...,True,78.024,0,True,2024-07-23T22:30:27.698024,NaN,NaN,NaN,NaN,NaN
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-23,True,1,0,0,0.0858371549,0,56134.584,4780.2938888333,...,2024-07-22T22:35:00.000000,70.218,0,True,2024-07-24T23:16:03.747212,NaN,NaN,NaN,NaN,NaN


In [485]:
# Get the current working directory
current_directory = os.getcwd()
print('current_directory: ', current_directory)

# Specify the file name
file_name = 'dtc_Summary_df.csv'

# Create the full file path
file_path = os.path.join(current_directory, file_name)

# Save the DataFrame to a CSV file
summary_df.to_csv(file_path, index=False)

# Display a message indicating the file has been saved
print(f"DataFrame has been saved to {file_path}")

current_directory:  c:\Users\programming.com\Downloads\HR_Project\ver0
DataFrame has been saved to c:\Users\programming.com\Downloads\HR_Project\ver0\dtc_Summary_df.csv


In [486]:
list(summary_df.columns.sort_values())

['active_time',
 'activity_automotive_count',
 'activity_automotive_max',
 'activity_automotive_mean',
 'activity_automotive_min',
 'activity_automotive_std',
 'activity_automotive_sum',
 'activity_cycling_count',
 'activity_cycling_max',
 'activity_cycling_mean',
 'activity_cycling_min',
 'activity_cycling_std',
 'activity_cycling_sum',
 'activity_percent',
 'activity_reboot_count',
 'activity_reboot_max',
 'activity_reboot_mean',
 'activity_reboot_min',
 'activity_reboot_std',
 'activity_reboot_sum',
 'activity_running_count',
 'activity_running_max',
 'activity_running_mean',
 'activity_running_min',
 'activity_running_std',
 'activity_running_sum',
 'activity_stationary_count',
 'activity_stationary_max',
 'activity_stationary_mean',
 'activity_stationary_min',
 'activity_stationary_std',
 'activity_stationary_sum',
 'activity_tilting_count',
 'activity_tilting_max',
 'activity_tilting_mean',
 'activity_tilting_min',
 'activity_tilting_std',
 'activity_tilting_sum',
 'activity_unkn

In [487]:
summary_df = summary_df[[
    'user_id',
 'date',
 'sleep_start',
 'sleep_end',
 'home_lat',
 'home_lon',

 ]]

summary_df.head()

,user_id,date,sleep_start,sleep_end,home_lat,home_lon
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-19,True,True,True,True
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-20,True,True,True,True
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-21,2024-07-20T21:50:00.000000,2024-07-21T09:15:00.000000,True,True
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-22,True,True,True,True
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-23,2024-07-22T22:35:00.000000,2024-07-23T09:25:00.000000,True,True


In [488]:
summary_df.shape

(5879, 6)

In [489]:
summary_df.user_id.nunique()

62

In [490]:
summary_df.columns

Index(['user_id', 'date', 'sleep_start', 'sleep_end', 'home_lat', 'home_lon'], dtype='object')

In [491]:
merged_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3


In [492]:
merged_df.shape

(685, 19)

In [493]:
merged_df.columns

Index(['user_id', 'timestamp', 'gender_identity', 'sex_at_birth',
       'sexual_orientation', 'ethnicity', 'survey_name', 'phq8_q1', 'phq8_q2',
       'phq8_q3', 'phq8_q4', 'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8',
       'phq9_q9', 'phq9_intro', 'phq9_outro', 'phq_total'],
      dtype='object')

In [494]:
# Get unique user_ids from both dataframes
merged_user_ids = set(merged_df['user_id'].unique())
summary_user_ids = set(summary_df['user_id'].unique())

# Find the additional user_ids present in summary_df but not in merged_df
additional_user_ids = summary_user_ids - merged_user_ids

# Filter summary_df for these additional user_ids (all columns will be returned)
additional_summary_df = summary_df[summary_df['user_id'].isin(additional_user_ids)]

print("Additional user_ids and their details from summary_df:")
additional_summary_df

Additional user_ids and their details from summary_df:


,user_id,date,sleep_start,sleep_end,home_lat,home_lon
0,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-19,True,True,True,True
1,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-20,True,True,True,True
2,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-21,2024-07-20T21:50:00.000000,2024-07-21T09:15:00.000000,True,True
3,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-22,True,True,True,True
4,us-east-1:db841a21-c463-c74c-9f67-6288c1827fca,2024-07-23,2024-07-22T22:35:00.000000,2024-07-23T09:25:00.000000,True,True
...,...,...,...,...,...,...
5429,us-east-1:db841a21-c4b8-cda3-f589-9199236d003c,2025-01-10,True,True,True,True
5430,us-east-1:db841a21-c4b8-cda3-f589-9199236d003c,2025-01-11,True,True,True,True
5431,us-east-1:db841a21-c4b8-cda3-f589-9199236d003c,2025-01-12,True,True,True,True
5432,us-east-1:db841a21-c4b8-cda3-f589-9199236d003c,2025-01-13,True,True,True,True


In [495]:
# Merge merged_df and summary_df based on the common 'user_id' column using an inner join
common_df = pd.merge(merged_df, summary_df, on='user_id', how='inner')
common_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,...,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total,date,sleep_start,sleep_end,home_lat,home_lon
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,...,0,0,0,0,5,2024-11-24,True,True,True,True
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,...,0,0,0,0,5,2024-11-25,2024-11-24T22:50:00.000000,2024-11-25T07:15:00.000000,True,True
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,...,0,0,0,0,5,2024-11-26,True,True,True,True
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,...,0,0,0,0,5,2024-11-27,True,True,True,True
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,...,0,0,0,0,5,2024-11-28,True,True,True,True


In [496]:
print("Shape of the common dataframe:", common_df.shape)

Shape of the common dataframe: (64505, 24)


In [497]:
a = common_df[(common_df.user_id == 'us-east-1:db841a21-c44c-cae0-4d10-3576737ed654') & (common_df.home_lat == True)]
a.date.value_counts()

date
2024-11-24    16
2024-11-25    16
2024-12-22    16
2024-12-23    16
2024-12-24    16
2024-12-25    16
2024-12-26    16
2024-12-27    16
2024-12-28    16
2024-12-29    16
2024-12-30    16
2024-12-31    16
2025-01-01    16
2025-01-02    16
2025-01-03    16
2025-01-04    16
2025-01-05    16
2025-01-06    16
2025-01-07    16
2025-01-08    16
2025-01-09    16
2025-01-10    16
2025-01-11    16
2025-01-12    16
2025-01-13    16
2024-12-21    16
2024-12-20    16
2024-12-19    16
2024-12-06    16
2024-11-26    16
2024-11-27    16
2024-11-28    16
2024-11-29    16
2024-11-30    16
2024-12-01    16
2024-12-02    16
2024-12-03    16
2024-12-04    16
2024-12-05    16
2024-12-07    16
2024-12-18    16
2024-12-08    16
2024-12-09    16
2024-12-10    16
2024-12-11    16
2024-12-12    16
2024-12-13    16
2024-12-14    16
2024-12-15    16
2024-12-16    16
2024-12-17    16
2025-01-14    16
Name: count, dtype: int64

# Final Analysis begins here

In [498]:
merged_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3


In [499]:
merged_df.shape

(685, 19)

In [500]:
merged_df.user_id.nunique()

52

In [503]:
merged_df[['user_id', 'gender_identity',	'sex_at_birth', 'sexual_orientation']]

,user_id,gender_identity,sex_at_birth,sexual_orientation
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,female,female,straight_or_heterosexual
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN,NaN,NaN
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN,NaN,NaN
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,female,female,straight_or_heterosexual
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,female,female,straight_or_heterosexual
...,...,...,...,...
680,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,female,female,straight_or_heterosexual
681,us-east-1:db841a21-c42d-c800-139a-89f64634e260,female,female,straight_or_heterosexual
682,us-east-1:db841a21-c42d-c800-139a-89f64634e260,NaN,NaN,NaN
683,us-east-1:db841a21-c42d-c800-139a-89f64634e260,NaN,NaN,NaN


In [504]:
merged_df.gender_identity.value_counts()

gender_identity
female     220
male       138
decline      2
Name: count, dtype: int64

In [505]:
merged_df.sex_at_birth.value_counts()

sex_at_birth
female     222
male       136
decline      2
Name: count, dtype: int64

In [507]:
merged_df.sexual_orientation.value_counts()

sexual_orientation
straight_or_heterosexual    292
bisexual                     30
queer_pan_questioning        18
lesbian_or_gay               12
decline                       8
Name: count, dtype: int64

In [508]:
new_df = merged_df[['user_id','gender_identity']]
new_df

,user_id,gender_identity
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,female
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,female
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,female
...,...,...
680,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,female
681,us-east-1:db841a21-c42d-c800-139a-89f64634e260,female
682,us-east-1:db841a21-c42d-c800-139a-89f64634e260,NaN
683,us-east-1:db841a21-c42d-c800-139a-89f64634e260,NaN


In [509]:
new_df.gender_identity.value_counts()

gender_identity
female     220
male       138
decline      2
Name: count, dtype: int64

In [515]:
new_df[new_df.gender_identity.isna() == True]

,user_id,gender_identity
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
5,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
6,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
9,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,NaN
...,...,...
675,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,NaN
678,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,NaN
679,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,NaN
682,us-east-1:db841a21-c42d-c800-139a-89f64634e260,NaN


In [511]:
220+138+2+325

685

In [518]:
merged_df.user_id.nunique()

52

In [528]:
new_df = merged_df[['user_id','gender_identity']]
df_unique = new_df.drop_duplicates(subset=['user_id'])

sex_counts = df_unique['gender_identity'].value_counts()
print('Sex Counts: ', sex_counts)

Sex Counts:  gender_identity
female     30
male       21
decline     1
Name: count, dtype: int64


In [530]:
sex_percentages = (sex_counts / len(df_unique)) * 100

print('Sex Percentages: ', sex_percentages.round(2))

Sex Percentages:  gender_identity
female     57.69
male       40.38
decline     1.92
Name: count, dtype: float64


In [531]:
merged_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total
0,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
1,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
2,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
3,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-02T20:33:56.380000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,1,0,2,0,1,0,0,0,0,0,5
4,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,2024-12-10T14:50:00.556000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,0,0,1,0,1,0,0,0,0,0,3


In [542]:
merged_df[ ['phq8_q1', 'phq8_q2', 'phq8_q3', 'phq8_q4', 
                  'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8', 'phq8_total']]

,phq8_q1,phq8_q2,phq8_q3,phq8_q4,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq8_total
21,1,1,2,2,2,2,2,2,14
22,1,1,2,2,2,2,2,2,14
23,1,1,2,2,2,2,2,2,14
24,1,1,2,2,2,2,2,2,14
319,1,0,0,1,0,0,2,1,5
...,...,...,...,...,...,...,...,...,...
553,3,3,1,3,3,3,2,2,20
554,3,3,1,3,3,3,2,2,20
555,3,3,1,3,3,3,2,2,20
556,3,3,1,3,3,3,2,2,20


In [544]:
# Assuming your dataframe is named df
columns_to_sum = ['phq8_q1', 'phq8_q2', 'phq8_q3', 'phq8_q4', 
                  'phq8_q5', 'phq8_q6', 'phq8_q7', 'phq8_q8']


# Convert each column to numeric, coercing errors to NaN
for col in columns_to_sum:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

# Now, create the new column 'phq8_total' as the row-wise sum of the specified columns
merged_df['phq8_total'] = merged_df[columns_to_sum].sum(axis=1)

merged_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,...,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total,ts,phq8_total
21,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,female,female,straight_or_heterosexual,[{'S': 'hispanic_or_latino'}],phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
22,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
23,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
24,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,female,female,straight_or_heterosexual,[{'S': 'hispanic_or_latino'}],phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
319,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,2024-11-21T14:49:12.334000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,0,0,...,0,0,2,1,0,0,0,5,2024-11-21 14:49:12.334000-05:00,5


In [ ]:

#  if you have a ts column, sort by user and time
if 'timestamp' in merged_df.columns:
    merged_df['ts'] = pd.to_datetime(merged_df['timestamp'])
    merged_df = merged_df.sort_values(by=['user_id', 'timestamp'])

# Get first and last PHQ8 score for each user
first_scores = merged_df.groupby('user_id').first()['phq8_total']
last_scores = merged_df.groupby('user_id').last()['phq8_total']

# Combine into a new DataFrame
phq_df_1 = pd.DataFrame({
    'phq8_before': first_scores,
    'phq8_after': last_scores
}).reset_index()

# Removing the users who have submitted just once 
counts = merged_df.groupby('user_id')[['phq8_total']].nunique().reset_index()
counts= pd.DataFrame(counts)
counts.columns
counts= counts[counts.phq8_total == 1]
phq_df =  phq_df_1[~phq_df_1['user_id'].isin(counts['user_id'])]

# Compute change
phq_df['change'] = phq_df['phq8_after'] - phq_df['phq8_before']

# Categorize users
no_change = phq_df[phq_df['change'] == 0]
improved = phq_df[phq_df['change'] < 0]
declined = phq_df[phq_df['change'] > 0]

# Summary stats
total = len(phq_df)
no_change_count = len(no_change)
improved_count = len(improved)
declined_count = len(declined)

no_change_percent = round((no_change_count / total) * 100, 1)
improved_percent = round((improved_count / total) * 100, 1)
declined_percent = round((declined_count / total) * 100, 1)

avg_improvement = round(abs(improved['change'].mean()), 2) if improved_count > 0 else 0
avg_decline = round(abs(declined['change'].mean()), 2) if declined_count > 0 else 0

# Output
print(f"{no_change_count}/{total} users had no change ({no_change_percent}%)")
print(f"{improved_count}/{total} users improved ({improved_percent}%) with an average change in PHQ-8 of {avg_improvement} points")
print(f"{declined_count}/{total} users declined ({declined_percent}%) with an average change in PHQ-8 of {avg_decline} points")

1/30 users had no change (3.3%)
19/30 users improved (63.3%) with an average change in PHQ-8 of 4.63 points
10/30 users declined (33.3%) with an average change in PHQ-8 of 2.8 points


In [549]:
counts = merged_df.groupby('user_id')[['phq8_total']].nunique().reset_index()
counts= pd.DataFrame(counts)
counts

,user_id,phq8_total
0,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,1
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,3
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,3
3,us-east-1:db841a21-c425-c26d-799b-adfd25f4c10f,2
4,us-east-1:db841a21-c425-c7f0-ffcb-4f7feec2627c,1
5,us-east-1:db841a21-c428-cd84-e285-398cf452c217,1
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,6
7,us-east-1:db841a21-c42d-c800-139a-89f64634e260,1
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,3
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,2


In [550]:
counts= counts[counts.phq8_total == 1]
counts

,user_id,phq8_total
0,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,1
4,us-east-1:db841a21-c425-c7f0-ffcb-4f7feec2627c,1
5,us-east-1:db841a21-c428-cd84-e285-398cf452c217,1
7,us-east-1:db841a21-c42d-c800-139a-89f64634e260,1
17,us-east-1:db841a21-c45f-cdc9-e7ac-ccae31af5676,1
19,us-east-1:db841a21-c468-cd6d-f7ab-edb30daa7e62,1
22,us-east-1:db841a21-c472-c904-9736-4281ee48e8a7,1
23,us-east-1:db841a21-c486-c9e8-a8f3-129cea57880e,1
24,us-east-1:db841a21-c48d-c8f8-a4d2-0c85eccbbb2f,1
30,us-east-1:db841a21-c4a5-c92c-6216-bfd4c217a3cf,1


In [551]:
len(counts)

22

In [552]:

phq_df =  phq_df_1[~phq_df_1['user_id'].isin(counts['user_id'])]
phq_df.shape

(30, 3)

## Final Ask from Sarah (phq > 10 and phq >5)

In [553]:

# Optional: Sort by timestamp
if 'timestamp' in merged_df.columns:
    merged_df['ts'] = pd.to_datetime(merged_df['timestamp'])
    merged_df = merged_df.sort_values(by=['user_id', 'ts'])

# Get first and last PHQ score for each user
first_scores = merged_df.groupby('user_id').first()['phq8_total']
last_scores = merged_df.groupby('user_id').last()['phq8_total']

# Combine into new dataframe
phq_df_1 = pd.DataFrame({
    'user_id': first_scores.index,
    'phq8_before': first_scores.values,
    'phq8_after': last_scores.values
})

# Remove users with only one PHQ entry
entry_counts = merged_df.groupby('user_id')['phq8_total'].nunique().reset_index()
single_entry_users = entry_counts[entry_counts['phq8_total'] == 1]['user_id']
phq_df = phq_df_1[~phq_df_1['user_id'].isin(single_entry_users)]

# Calculate change and percent change
phq_df['change'] = phq_df['phq8_after'] - phq_df['phq8_before']
phq_df['pct_change'] = (phq_df['change'] / phq_df['phq8_before']) * 100
print(f"The length of users is: {len(phq_df)}")

### ----------- FUNCTION TO PRINT METRICS FOR ANY SUBGROUP -----------
def analyze_change(sub_df, group_label):
    total_users = len(sub_df)
    if total_users == 0:
        print(f"\n{group_label}: No users in this group.")
        return
    

    # Decreased group
    decreased = sub_df[sub_df['change'] < 0]
    decreased_count = len(decreased)
    decreased_pct = round((decreased_count / total_users) * 100, 1)
    avg_pct_decrease = round(abs(decreased['pct_change'].mean()), 1) if decreased_count > 0 else 0
    avg_point_decrease = round(abs(decreased['change'].mean()), 2) if decreased_count > 0 else 0

    # Increased group
    increased = sub_df[sub_df['change'] > 0]
    increased_count = len(increased)
    increased_pct = round((increased_count / total_users) * 100, 1)
    avg_pct_increase = round(increased['pct_change'].mean(), 1) if increased_count > 0 else 0
    avg_point_increase = round(increased['change'].mean(), 2) if increased_count > 0 else 0
        
    # No change
    nc = sub_df[sub_df['change'] == 0]
    nc_count = len(nc)
    nc_pct = round((nc_count / total_users) * 100, 1)
    avg_pct_nc = round(nc['pct_change'].mean(), 1) if nc_count > 0 else 0
    avg_point_nc = round(nc['change'].mean(), 2) if nc_count > 0 else 0

    # Print Results
    print(f"\n📊 {group_label} (n = {total_users})")
    print(f"- {decreased_count} users ({decreased_pct}%) decreased")
    print(f"   ↪ Avg % decrease: {avg_pct_decrease}%, Avg point decrease: {avg_point_decrease}")

    print(f"- {increased_count} users ({increased_pct}%) increased")
    print(f"   ↪ Avg % increase: {avg_pct_increase}%, Avg point increase: {avg_point_increase}")

    print(f"- {nc_count} users ({nc_pct}%) has no change")
    print(f"   ↪ Avg % of no change users: {avg_pct_nc}%, Avg point of no change users: {avg_point_nc}")

### ----------- ANALYZE BOTH GROUPS -----------

# Group A: Users with baseline PHQ >10
group_a = phq_df[phq_df['phq8_before'] >= 10]
analyze_change(group_a, "Users with baseline PHQ8_total >= 10")

# Group B: Users with baseline PHQ >5
group_b = phq_df[phq_df['phq8_before'] >= 5]
analyze_change(group_b, "Users with baseline PHQ8_total >= 5")

The length of users is: 30

📊 Users with baseline PHQ8_total >= 10 (n = 13)
- 11 users (84.6%) decreased
   ↪ Avg % decrease: 45.5%, Avg point decrease: 6.36
- 2 users (15.4%) increased
   ↪ Avg % increase: 16.1%, Avg point increase: 2.5
- 0 users (0.0%) has no change
   ↪ Avg % of no change users: 0%, Avg point of no change users: 0

📊 Users with baseline PHQ8_total >= 5 (n = 25)
- 18 users (72.0%) decreased
   ↪ Avg % decrease: 41.0%, Avg point decrease: 4.67
- 7 users (28.0%) increased
   ↪ Avg % increase: 45.3%, Avg point increase: 3.0
- 0 users (0.0%) has no change
   ↪ Avg % of no change users: 0%, Avg point of no change users: 0


In [555]:

# Optional: Sort by timestamp
if 'timestamp' in merged_df.columns:
    merged_df['ts'] = pd.to_datetime(merged_df['timestamp'])
    merged_df = merged_df.sort_values(by=['user_id', 'ts'])

merged_df.head()

,user_id,timestamp,gender_identity,sex_at_birth,sexual_orientation,ethnicity,survey_name,phq8_q1,phq8_q2,phq8_q3,...,phq8_q5,phq8_q6,phq8_q7,phq8_q8,phq9_q9,phq9_intro,phq9_outro,phq_total,ts,phq8_total
21,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,female,female,straight_or_heterosexual,[{'S': 'hispanic_or_latino'}],phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
22,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
23,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,NaN,NaN,NaN,NaN,phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
24,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,2025-01-04T13:05:28.612000-05:00,female,female,straight_or_heterosexual,[{'S': 'hispanic_or_latino'}],phq9_with_outro,1,1,2,...,2,2,2,2,1,0,0,15,2025-01-04 13:05:28.612000-05:00,14
319,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,2024-11-21T14:49:12.334000-05:00,female,female,straight_or_heterosexual,[{'S': 'black_or_african_american'}],phq9_with_outro,1,0,0,...,0,0,2,1,0,0,0,5,2024-11-21 14:49:12.334000-05:00,5


In [556]:

# Get first and last PHQ score for each user
first_scores = merged_df.groupby('user_id').first()['phq8_total']
last_scores = merged_df.groupby('user_id').last()['phq8_total']


In [562]:

# Combine into new dataframe
phq_df_1 = pd.DataFrame({
    'user_id': first_scores.index,
    'phq8_before': first_scores.values,
    'phq8_after': last_scores.values
})
phq_df_1

,user_id,phq8_before,phq8_after
0,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,14,14
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,5,4
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,5,12
3,us-east-1:db841a21-c425-c26d-799b-adfd25f4c10f,0,1
4,us-east-1:db841a21-c425-c7f0-ffcb-4f7feec2627c,4,4
5,us-east-1:db841a21-c428-cd84-e285-398cf452c217,7,7
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12
7,us-east-1:db841a21-c42d-c800-139a-89f64634e260,7,7
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23


In [566]:

# Remove users with only one PHQ entry
entry_counts = merged_df.groupby('user_id')['phq8_total'].nunique().reset_index()
single_entry_users = entry_counts[entry_counts['phq8_total'] == 1]['user_id']
phq_df = phq_df_1[~phq_df_1['user_id'].isin(single_entry_users)]


In [567]:
entry_counts

,user_id,phq8_total
0,us-east-1:db841a21-c406-cbe8-8908-c7904aafe112,1
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,3
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,3
3,us-east-1:db841a21-c425-c26d-799b-adfd25f4c10f,2
4,us-east-1:db841a21-c425-c7f0-ffcb-4f7feec2627c,1
5,us-east-1:db841a21-c428-cd84-e285-398cf452c217,1
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,6
7,us-east-1:db841a21-c42d-c800-139a-89f64634e260,1
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,3
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,2


In [568]:
single_entry_users

0     us-east-1:db841a21-c406-cbe8-8908-c7904aafe112
4     us-east-1:db841a21-c425-c7f0-ffcb-4f7feec2627c
5     us-east-1:db841a21-c428-cd84-e285-398cf452c217
7     us-east-1:db841a21-c42d-c800-139a-89f64634e260
17    us-east-1:db841a21-c45f-cdc9-e7ac-ccae31af5676
19    us-east-1:db841a21-c468-cd6d-f7ab-edb30daa7e62
22    us-east-1:db841a21-c472-c904-9736-4281ee48e8a7
23    us-east-1:db841a21-c486-c9e8-a8f3-129cea57880e
24    us-east-1:db841a21-c48d-c8f8-a4d2-0c85eccbbb2f
30    us-east-1:db841a21-c4a5-c92c-6216-bfd4c217a3cf
31    us-east-1:db841a21-c4a7-ced2-1395-c814829d5426
33    us-east-1:db841a21-c4aa-c659-c0e0-b1385645b311
34    us-east-1:db841a21-c4b6-c79a-ef9c-3cd3c42019a4
35    us-east-1:db841a21-c4bf-cce3-677d-6f70dd30dbb4
36    us-east-1:db841a21-c4c3-c1cf-6500-e8736fa04321
37    us-east-1:db841a21-c4c4-c57c-e3e4-f3ee8247d775
41    us-east-1:db841a21-c4cf-c0f9-210b-e311de6b881b
45    us-east-1:db841a21-c4dd-c85c-653f-1d01bc283a75
46    us-east-1:db841a21-c4e8-ca31-8042-c2617c

In [569]:
phq_df

,user_id,phq8_before,phq8_after
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,5,4
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,5,12
3,us-east-1:db841a21-c425-c26d-799b-adfd25f4c10f,0,1
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23
10,us-east-1:db841a21-c436-cab5-9903-2c5cb1c4394c,4,0
11,us-east-1:db841a21-c445-ccbf-ecdf-8df123e10120,12,6
12,us-east-1:db841a21-c44b-c828-4fea-65f5b59027a2,0,4
13,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,5,2


In [ ]:
# Calculate change and percent change
phq_df['change'] = phq_df['phq8_after'] - phq_df['phq8_before']
phq_df['pct_change'] = (phq_df['change'] / phq_df['phq8_before']) * 100
phq_df

,user_id,phq8_before,phq8_after,change,pct_change
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,5,4,-1,-20.000000
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,5,12,7,140.000000
3,us-east-1:db841a21-c425-c26d-799b-adfd25f4c10f,0,1,1,inf
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12,-3,-20.000000
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9,-7,-43.750000
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23,-1,-4.166667
10,us-east-1:db841a21-c436-cab5-9903-2c5cb1c4394c,4,0,-4,-100.000000
11,us-east-1:db841a21-c445-ccbf-ecdf-8df123e10120,12,6,-6,-50.000000
12,us-east-1:db841a21-c44b-c828-4fea-65f5b59027a2,0,4,4,inf
13,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,5,2,-3,-60.000000


In [571]:
print(f"The length of users is: {len(phq_df)}")


The length of users is: 30


In [ ]:

### ----------- FUNCTION TO PRINT METRICS FOR ANY SUBGROUP -----------
def analyze_change(sub_df, group_label):
    total_users = len(sub_df) #13
    if total_users == 0:
        print(f"\n{group_label}: No users in this group.")
        return
    

    # Decreased group
    decreased = sub_df[sub_df['change'] < 0]
    decreased_count = len(decreased) #11
    decreased_pct = round((decreased_count / total_users) * 100, 1) #86%
    avg_pct_decrease = round(abs(decreased['pct_change'].mean()), 1) if decreased_count > 0 else 0
    avg_point_decrease = round(abs(decreased['change'].mean()), 2) if decreased_count > 0 else 0

    # Increased group
    increased = sub_df[sub_df['change'] > 0]
    increased_count = len(increased)
    increased_pct = round((increased_count / total_users) * 100, 1)
    avg_pct_increase = round(increased['pct_change'].mean(), 1) if increased_count > 0 else 0
    avg_point_increase = round(increased['change'].mean(), 2) if increased_count > 0 else 0
        
    # No change
    nc = sub_df[sub_df['change'] == 0]
    nc_count = len(nc)
    nc_pct = round((nc_count / total_users) * 100, 1)
    avg_pct_nc = round(nc['pct_change'].mean(), 1) if nc_count > 0 else 0
    avg_point_nc = round(nc['change'].mean(), 2) if nc_count > 0 else 0

    # Print Results
    print(f"\n📊 {group_label} (n = {total_users})")
    print(f"- {decreased_count} users ({decreased_pct}%) decreased")
    print(f"   ↪ Avg % decrease: {avg_pct_decrease}%, Avg point decrease: {avg_point_decrease}")

    print(f"- {increased_count} users ({increased_pct}%) increased")
    print(f"   ↪ Avg % increase: {avg_pct_increase}%, Avg point increase: {avg_point_increase}")

    print(f"- {nc_count} users ({nc_pct}%) has no change")
    print(f"   ↪ Avg % of no change users: {avg_pct_nc}%, Avg point of no change users: {avg_point_nc}")


In [ ]:

### ----------- ANALYZE BOTH GROUPS -----------

# Group A: Users with baseline PHQ >10
group_a = phq_df[phq_df['phq8_before'] >= 10]
analyze_change(group_a, "Users with baseline PHQ8_total >= 10")


In [577]:

# Group A: Users with baseline PHQ >10
group_a = phq_df[phq_df['phq8_before'] >= 10]
group_a

,user_id,phq8_before,phq8_after,change,pct_change
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12,-3,-20.000000
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9,-7,-43.750000
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23,-1,-4.166667
11,us-east-1:db841a21-c445-ccbf-ecdf-8df123e10120,12,6,-6,-50.000000
15,us-east-1:db841a21-c45b-cbcc-6e86-28bf1f056f0e,15,7,-8,-53.333333
18,us-east-1:db841a21-c461-c4b0-2db3-d0f5d28948d0,10,2,-8,-80.000000
20,us-east-1:db841a21-c46f-cdaa-fc8e-68406d727f96,18,19,1,5.555556
27,us-east-1:db841a21-c49f-c356-a9a0-ee2fe8537f83,15,19,4,26.666667
29,us-east-1:db841a21-c4a3-c353-85d8-b42a69a1f5d4,19,10,-9,-47.368421
38,us-east-1:db841a21-c4ca-ccd6-40dd-f8c1991cc0ad,11,6,-5,-45.454545


In [578]:
total_users= len(group_a)
total_users

13

In [579]:
group_a[group_a['change'] < 0]

,user_id,phq8_before,phq8_after,change,pct_change
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12,-3,-20.000000
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9,-7,-43.750000
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23,-1,-4.166667
11,us-east-1:db841a21-c445-ccbf-ecdf-8df123e10120,12,6,-6,-50.000000
15,us-east-1:db841a21-c45b-cbcc-6e86-28bf1f056f0e,15,7,-8,-53.333333
18,us-east-1:db841a21-c461-c4b0-2db3-d0f5d28948d0,10,2,-8,-80.000000
29,us-east-1:db841a21-c4a3-c353-85d8-b42a69a1f5d4,19,10,-9,-47.368421
38,us-east-1:db841a21-c4ca-ccd6-40dd-f8c1991cc0ad,11,6,-5,-45.454545
39,us-east-1:db841a21-c4ce-c0c6-34fd-af71468f2c40,11,5,-6,-54.545455
40,us-east-1:db841a21-c4ce-cb7d-1ced-d47d1fd96b1a,15,6,-9,-60.000000


In [580]:
decreased = group_a[group_a['change'] < 0]
decreased_count= len(decreased)

In [582]:
decreased_pct = round((decreased_count / total_users) * 100, 1)
decreased_pct

84.6

In [583]:
(11/13)*100

84.61538461538461

In [588]:
decreased['pct_change'].mean()

-45.520334928229666

In [584]:
avg_pct_decrease = round(abs(decreased['pct_change'].mean()), 1) if decreased_count > 0 else 0
avg_pct_decrease

45.5

In [591]:
decreased['change'].mean()

-6.363636363636363

In [589]:
avg_point_decrease = round(abs(decreased['change'].mean()), 2) if decreased_count > 0 else 0
avg_point_decrease

6.36

In [592]:
increased = group_a[group_a['change'] > 0]
increased

,user_id,phq8_before,phq8_after,change,pct_change
20,us-east-1:db841a21-c46f-cdaa-fc8e-68406d727f96,18,19,1,5.555556
27,us-east-1:db841a21-c49f-c356-a9a0-ee2fe8537f83,15,19,4,26.666667


In [593]:
increased_count = len(increased)
increased_count

2

In [594]:
increased_pct = round((increased_count / total_users) * 100, 1)
increased_pct

15.4

In [595]:
2/13*100

15.384615384615385

In [601]:
increased['pct_change']

20     5.555556
27    26.666667
Name: pct_change, dtype: float64

In [602]:
(5.555 + 26.66)/2

16.1075

In [600]:
increased['pct_change'].mean()

16.11111111111111

In [596]:
avg_pct_increase = round(increased['pct_change'].mean(), 1) if increased_count > 0 else 0
avg_pct_increase

16.1

In [604]:
increased['change'].mean()

2.5

In [605]:
increased['change']

20    1
27    4
Name: change, dtype: int64

In [606]:
(1+4)/2

2.5

In [597]:
avg_point_increase = round(increased['change'].mean(), 2) if increased_count > 0 else 0
avg_point_increase

2.5

In [608]:
nc = group_a[group_a['change'] == 0]
nc

,user_id,phq8_before,phq8_after,change,pct_change


In [609]:
nc_count = len(nc)
nc_count

0

In [610]:
nc_pct = round((nc_count / total_users) * 100, 1)
nc_pct

0.0

In [611]:
avg_pct_nc = round(nc['pct_change'].mean(), 1) if nc_count > 0 else 0
avg_pct_nc

0

In [612]:
avg_point_nc = round(nc['change'].mean(), 2) if nc_count > 0 else 0
avg_point_nc 

0

In [613]:

# Group B: Users with baseline PHQ >5
group_b = phq_df[phq_df['phq8_before'] >= 5]
analyze_change(group_b, "Users with baseline PHQ8_total >= 5")


📊 Users with baseline PHQ8_total >= 5 (n = 25)
- 18 users (72.0%) decreased
   ↪ Avg % decrease: 41.0%, Avg point decrease: 4.67
- 7 users (28.0%) increased
   ↪ Avg % increase: 45.3%, Avg point increase: 3.0
- 0 users (0.0%) has no change
   ↪ Avg % of no change users: 0%, Avg point of no change users: 0


In [616]:

# Group A: Users with baseline PHQ >=5
group_a = phq_df[phq_df['phq8_before'] >= 5]
group_a

,user_id,phq8_before,phq8_after,change,pct_change
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,5,4,-1,-20.000000
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,5,12,7,140.000000
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12,-3,-20.000000
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9,-7,-43.750000
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23,-1,-4.166667
11,us-east-1:db841a21-c445-ccbf-ecdf-8df123e10120,12,6,-6,-50.000000
13,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,5,2,-3,-60.000000
14,us-east-1:db841a21-c452-cb1e-cccf-581320eb1e27,5,4,-1,-20.000000
15,us-east-1:db841a21-c45b-cbcc-6e86-28bf1f056f0e,15,7,-8,-53.333333
16,us-east-1:db841a21-c45c-c1e1-aa7f-9019b93342dc,6,7,1,16.666667


In [617]:
total_users= len(group_a)
total_users

25

In [618]:
group_a[group_a['change'] < 0]

,user_id,phq8_before,phq8_after,change,pct_change
1,us-east-1:db841a21-c408-c990-452a-692d8dfd6105,5,4,-1,-20.000000
6,us-east-1:db841a21-c42d-c7a4-27e2-fac02245e5eb,15,12,-3,-20.000000
8,us-east-1:db841a21-c432-cdea-1dd2-ebf590fe5fa2,16,9,-7,-43.750000
9,us-east-1:db841a21-c433-c0e3-61a7-dc62e500aa3b,24,23,-1,-4.166667
11,us-east-1:db841a21-c445-ccbf-ecdf-8df123e10120,12,6,-6,-50.000000
13,us-east-1:db841a21-c44c-cae0-4d10-3576737ed654,5,2,-3,-60.000000
14,us-east-1:db841a21-c452-cb1e-cccf-581320eb1e27,5,4,-1,-20.000000
15,us-east-1:db841a21-c45b-cbcc-6e86-28bf1f056f0e,15,7,-8,-53.333333
18,us-east-1:db841a21-c461-c4b0-2db3-d0f5d28948d0,10,2,-8,-80.000000
21,us-east-1:db841a21-c471-c321-a10d-3679fbf43562,8,5,-3,-37.500000


In [620]:
decreased = group_a[group_a['change'] < 0]
decreased_count= len(decreased)
decreased_count

18

In [622]:
(18/25)*100

72.0

In [621]:
decreased_pct = round((decreased_count / total_users) * 100, 1)
decreased_pct

72.0

In [627]:
decreased['pct_change'].mean()

-41.01242690058479

In [ ]:
avg_pct_decrease = round(abs(decreased['pct_change'].mean()), 1) if decreased_count > 0 else 0
avg_pct_decrease

45.5

In [628]:
decreased['change'].mean()

-4.666666666666667

In [629]:
avg_point_decrease = round(abs(decreased['change'].mean()), 2) if decreased_count > 0 else 0
avg_point_decrease

4.67

In [630]:
increased = group_a[group_a['change'] > 0]
increased

,user_id,phq8_before,phq8_after,change,pct_change
2,us-east-1:db841a21-c40d-cfe1-ab63-0f575df1b398,5,12,7,140.000000
16,us-east-1:db841a21-c45c-c1e1-aa7f-9019b93342dc,6,7,1,16.666667
20,us-east-1:db841a21-c46f-cdaa-fc8e-68406d727f96,18,19,1,5.555556
26,us-east-1:db841a21-c494-c766-49ef-f45ba6c48f37,5,7,2,40.000000
27,us-east-1:db841a21-c49f-c356-a9a0-ee2fe8537f83,15,19,4,26.666667
42,us-east-1:db841a21-c4cf-c1ad-b90a-78145dd7ca9c,6,7,1,16.666667
50,us-east-1:db841a21-c4fc-c053-ff60-9e408e777bd3,7,12,5,71.428571


In [631]:
increased_count = len(increased)
increased_count

7

In [632]:
increased_pct = round((increased_count / total_users) * 100, 1)
increased_pct

28.0

In [633]:
7/25*100

28.000000000000004

In [637]:
increased['pct_change'].values

array([140.        ,  16.66666667,   5.55555556,  40.        ,
        26.66666667,  16.66666667,  71.42857143])

In [635]:
increased['pct_change'].mean()

45.28344671201813

In [638]:
(140 + 16.66666667 + 5.55555556 + 40 + 26.66666667 + 16.66666667 + 71.42857143)/7

45.28344671428572

In [639]:
increased['pct_change'].mean()

45.28344671201813

In [640]:
avg_pct_increase = round(increased['pct_change'].mean(), 1) if increased_count > 0 else 0
avg_pct_increase

45.3

In [641]:
increased['change'].mean()

3.0

In [642]:
increased['change']

2     7
16    1
20    1
26    2
27    4
42    1
50    5
Name: change, dtype: int64

In [646]:
(7+1+1+2+4+1+5)/7

3.0

In [645]:
avg_point_increase = round(increased['change'].mean(), 2) if increased_count > 0 else 0
avg_point_increase

3.0

In [647]:
nc = group_a[group_a['change'] == 0]
nc

,user_id,phq8_before,phq8_after,change,pct_change


In [648]:
nc_count = len(nc)
nc_count

0

In [649]:
nc_pct = round((nc_count / total_users) * 100, 1)
nc_pct

0.0

In [650]:
avg_pct_nc = round(nc['pct_change'].mean(), 1) if nc_count > 0 else 0
avg_pct_nc

0

In [651]:
avg_point_nc = round(nc['change'].mean(), 2) if nc_count > 0 else 0
avg_point_nc 

0